# 10 - Retrain Final Model
 Retrain the selected model (SVM) on the best set of features (Fused featureset) using the full development data set (train + validation) and evaluate on the hold-out test set


In [ ]:
import datetime
import glob
import os
import numpy as np
import pandas as pd
import time

from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
start_time = time.perf_counter()

In [ ]:
from utils import set_seed, load_features_from_csv
from utils import print_time_since
from visualize import pred_scatterplot


from sklearn.model_selection import train_test_split, GroupShuffleSplit, GroupKFold, GridSearchCV
from sklearn.linear_model import ElasticNet
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.metrics import f1_score, roc_curve, roc_auc_score,recall_score,precision_score,accuracy_score, log_loss

from scipy.stats import loguniform, uniform, randint, pearsonr, spearmanr
from statsmodels.stats.multitest import multipletests


In [ ]:
from scipy.stats import mannwhitneyu

In [ ]:
import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.simplefilter("ignore", ConvergenceWarning)

In [ ]:
from gene_model import Calibration

In [ ]:
pd.set_option('display.precision', 3)

# Define Variables

In [ ]:
segment_order = ['CD45+', 'PanCK+']
geneset_order = ['Panel','HVG', 'SVG']
SCORE_STAT = 'neg_mean_squared_error'
CALIBRATION_FRACTION=0.15
seed=77

In [ ]:
model_lookup = {
    'SVM': SVR,
    'elasticnet': lambda: ElasticNet(random_state=seed)
}

# Define Utility Functions

In [ ]:
from dataclasses import dataclass
from typing import Dict, Tuple, List, Any, Iterable

In [ ]:
def plot_top_gene_preds(y_test, y_pred, genenames, output_path):
    num_genes = len(genenames)
    fig, axs = plt.subplots(5, 5, figsize=(15, 15), sharex=False, sharey=False)
    axs = axs.flatten()

    for i, genename in enumerate(genenames):
   
        sns.scatterplot(x=y_test[genename], y=y_pred[genename], ax=axs[i])
        plt.xlabel(f"True Expression ({genename})")
        plt.ylabel("Predicted Expression")
        plt.title(f"Actual vs Predicted for {genename}")
        plt.plot([y_test[:, i].min(), y_test[:, i].max()],
                [y_test[:, i].min(), y_test[:, i].max()], 'r--')
        plt.grid(True)

In [ ]:
def plot_and_save_gene_preds(y_test, y_pred, genenames, geneset, output_path):

    figpath = os.path.join(output_path,'figs')
    os.makedirs(figpath,exist_ok=True)
    
    for i, genename in enumerate(genenames):
        curr_date=str(datetime.date.today())
        curr_time=datetime.datetime.now().strftime('%H%M')
        filename=f'summaryplot-{genename}-{geneset}-{curr_date}_{curr_time}.png'

        # Optional: Plot one gene as example (e.g., gene 0)
        plt.figure(figsize=(6, 6))
        sns.scatterplot(x=y_test[:, i], y=y_pred[:, i])
        plt.xlabel(f"True Expression ({genename})")
        plt.ylabel("Predicted Expression")
        plt.title(f"Actual vs Predicted for {geneset} {genename}")
        plt.plot([y_test[:, i].min(), y_test[:, i].max()],
                [y_test[:, i].min(), y_test[:, i].max()], 'r--')
        plt.grid(True)
        plotname = f"scatter_{geneset}_{genename}.png".replace(" ", "_")
        #print(f"saving plot to {output_path}/{filename}")
        #print("os.path.exists:", os.path.exists(os.path.join(output_path, filename)))
        #print("Can write to output_path:", os.access(output_path, os.W_OK))

        try:
            plt.savefig(os.path.join(figpath,filename), dpi=300, bbox_inches='tight')
            #print(f"Saved plot to: {figpath}")
            
        except Exception as e:
            print(f"Failed to save plot ")
        #plt.show()
        plt.close()

In [ ]:
def concordance_correlation_coeff(y_true, y_pred):
    """Concordance Correlation Coefficient (CCC) between true and predicted values."""
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    mean_true = np.mean(y_true)
    mean_pred = np.mean(y_pred)
    var_true = np.var(y_true, ddof=0)
    var_pred = np.var(y_pred, ddof=0)
    cov = np.mean((y_true - mean_true) * (y_pred - mean_pred))
    #cov = np.cov(y_true, y_pred, ddof=1)[0, 1] # sample rather than population

    ccc = (2 * cov) / (var_true + var_pred + (mean_true - mean_pred) ** 2)
    return ccc

In [ ]:
def variance_ratio(y_true, y_pred, ddof=0):
    """
    Compute the ratio of the variance of predictions to that of true values.

    Parameters
    ----------
    y_true : array-like
        Ground-truth target values.
    y_pred : array-like
        Model predictions.
    ddof : int, default=1
        Delta degrees of freedom for variance (as in np.var).

    Returns
    -------
    float
        Variance ratio = Var(y_pred) / Var(y_true)
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    var_true = np.var(y_true, ddof=ddof)
    var_pred = np.var(y_pred, ddof=ddof)

    return var_pred / var_true

In [ ]:
def cliffs_delta(a, b):
    a = np.asarray(a)
    b = np.asarray(b)
    
    a = a[~np.isnan(a)]
    b = b[~np.isnan(b)]
    
    return (np.sum(a[:, None] > b[None, :]) -
            np.sum(a[:, None] < b[None, :])) / (len(a) * len(b))




In [ ]:
def affine_calibration(y_true, y_pred, eps: float = 1e-12):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    mu_y = np.mean(y_true)
    mu_hat = np.mean(y_pred)

    sigma_y = np.std(y_true, ddof=0)
    sigma_hat = np.std(y_pred, ddof=0)

    # ill-posed calibration: constant true or constant pred
    #if sigma_y < eps or sigma_hat < eps:
        # sensible fallback: predict the mean of the truth
    #    return 0.0, mu_y
    if sigma_y < eps or sigma_hat < 1e-3:
        return 1.0, 0.0  # skip calibration


    r = np.corrcoef(y_true, y_pred)[0, 1]
    if not np.isfinite(r):
        r = 0.0

    #alpha = np.sign(r) * (sigma_y / sigma_hat)
    alpha = sigma_y / sigma_hat
    beta = mu_y - alpha * mu_hat

    if not (np.isfinite(alpha) and np.isfinite(beta)):
        return 0.0, mu_y

    return alpha, beta


In [ ]:
def get_regression_scores(y_test, y_pred, geneset='HVG',segment='CD45+',feat_extr='uni', model='SVM',output_path=''):
    gene_names = list(y_test.columns)
    y_test = np.asarray(y_test)
    y_pred = np.asarray(y_pred)

    #for i, gene in enumerate(gene_names):
    #    print(f"SCORING {gene}",flush=True)

    # Compute per-gene (column) metrics
    mse = mean_squared_error(y_test, y_pred, multioutput='raw_values')
    rmse = mse ** 0.5
    mae = mean_absolute_error(y_test, y_pred, multioutput='raw_values')
    r2 = r2_score(y_test, y_pred, multioutput='raw_values')

    # Compute std of true expression (for nRMSE)
    std = np.std(y_test, axis=0, ddof=1)
    nrmse = rmse / std
    
    # Correlations per gene
    pearson_results = [pearsonr(y_test[:, i], y_pred[:, i]) for i in range(y_test.shape[1])]
    spearman_results = [spearmanr(y_test[:, i], y_pred[:, i]) for i in range(y_test.shape[1])]
    pearson_corrs, pearson_pvals = zip(*pearson_results)
    spearman_corrs, spearman_pvals = zip(*spearman_results)
    pearson_ci_highs = [res.confidence_interval().high for res in pearson_results]
    pearson_ci_lows = [res.confidence_interval().low for res in pearson_results]
    pearson_ci_widths = [high - low for high, low in zip(pearson_ci_highs, pearson_ci_lows)]
    
    # FDR correction =- Pearson
    #_, pearson_qvals, _, _ = multipletests(pearson_pvals, method='fdr_bh')
    p = np.asarray(pearson_pvals, dtype=float)      # your tuple/array of p-values
    q = np.full(p.shape, np.nan, dtype=float)       # prefill with NaN

    m = np.isfinite(p)                              # mask out NaNs/inf
    if m.any():
        q[m] = multipletests(p[m], method='fdr_bh')[1]

    pearson_qvals = q

    # FDR correction =- Pearson
    #_, pearson_qvals, _, _ = multipletests(spearman_pvals, method='fdr_bh')
    p = np.asarray(spearman_pvals, dtype=float)      # your tuple/array of p-values
    q = np.full(p.shape, np.nan, dtype=float)       # prefill with NaN

    m = np.isfinite(p)                              # mask out NaNs/inf
    if m.any():
        q[m] = multipletests(p[m], method='fdr_bh')[1]

    spearman_qvals = q
    #_, spearman_qvals, _, _ = multipletests(spearman_pvals, method='fdr_bh')
    #print(pearson_qvals)


    vr = [variance_ratio(y_test[:, i], y_pred[:, i]) for i in range(y_test.shape[1])]
    ccc = [concordance_correlation_coeff(y_test[:, i], y_pred[:, i]) for i in range(y_test.shape[1])]

    # Per-gene DataFrame
    n_genes = y_test.shape[1]
    per_gene_df = pd.DataFrame({
        'Gene': gene_names, #[f'Gene_{i}' for i in range(n_genes)],
        'MSE': mse,
        'RMSE': rmse,
        'nRMSE': nrmse,
        'MAE': mae,
        'R2': r2,
        'PearsonCorr': pearson_corrs,
        'PearsonPVals': pearson_pvals,
        'PearsonQVals': pearson_qvals,
        'PearsonCI_low': pearson_ci_lows,
        'PearsonCI_high': pearson_ci_highs,
        'SpearmanCorr': spearman_corrs,
        'SpearmanPVals': spearman_pvals,
        'SpearmanQVals': spearman_qvals,
        'VarianceRatio': vr,
        'ConcordanceCorr': ccc,
    })
    per_gene_df.insert(0, 'GeneSet', geneset)
    per_gene_df.insert(1, 'Segment', segment)
    per_gene_df.insert(2,'Feature_Extractor',feat_extr)
    per_gene_df.insert(3,'Model',model)

    # Summary row
    # Aggregate (mean)
    mean_metrics = {
        'MSE': mse.mean(),
        'RMSE': rmse.mean(),
        'nRMSE': np.nanmean(nrmse),
        'MAE': mae.mean(),
        'R2': r2.mean(),
        'PearsonCorr': np.nanmean(pearson_corrs),
        'PearsonPVals': np.median(pearson_pvals),
        'PearsonQVals': np.median(pearson_qvals),
        'PearsonCIWidth': np.median(pearson_ci_widths),
        'SpearmanCorr': np.nanmean(spearman_corrs),
        'SpearmanPVals': np.median(spearman_pvals),
        'SpearmanQVals': np.median(spearman_qvals),
        'VarianceRatio': np.nanmean(vr),
        'ConcordanceCorr': np.nanmean(ccc),

    }

    summary_df = pd.DataFrame([mean_metrics])
    summary_df.insert(0, 'GeneSet', geneset)
    summary_df.insert(1, 'Segment', segment)
    summary_df.insert(2,'Feature_Extractor',feat_extr)
    summary_df.insert(3,'Model',model)



    #plot_and_save_gene_preds(y_test, y_pred, gene_names,geneset,output_path)

    return summary_df, per_gene_df




In [ ]:
def train_final_regressor(model, X_train, y_train, gene_names, gene_params, verbose=False):
    if verbose: print(f"Training {len(gene_names)} models")
    tuned_models = []

    if model not in model_lookup:
        print(f"Unrecognized model type: {model}")
        return

    for i in range(y_train.shape[1]):

        #print(f"{i} - Training model for gene {gene_names[i]}")
        params = eval(gene_params[i])

        base_model = model_lookup[model]()
        model_instance  = base_model.set_params(**params)
        model_instance.fit(X_train, y_train.iloc[:, i])
        tuned_models.append(model_instance)
        
    return tuned_models

In [ ]:
def evaluate_final_models(models, X_test, y_test, 
                          geneset='HVG',segment='CD45+',feat_extr='uni',model='SVM',
                          output_path='',save=True,calibration=None, verbose=False):
    per_gene_scores_cal = None
    y_pred_cal = None
    if verbose: print("save", save)
    # Predict each gene separately
    y_pred = np.column_stack([model.predict(X_test) for model in models])



    # Compute metrics
    summary_scores, per_gene_scores = get_regression_scores(
        y_test,
        y_pred,
        geneset,segment,feat_extr,model,
        output_path=output_path
    )
    display(summary_scores.head())

    # post-hoc calibration
    if calibration:
        y_pred_cal = calibration.apply(y_pred)
            # Compute metrics
        summary_scores_cal, per_gene_scores_cal = get_regression_scores(
            y_test,
            y_pred_cal,
            geneset,segment,feat_extr,model,
            output_path=output_path
        )
        display(summary_scores_cal.head())



    if save:
        # Save per-gene metrics
        if verbose: print(f"saving to: {output_path}")
        per_gene_scores.to_csv(os.path.join(output_path,f"FINAL_per_gene_scores-{geneset}-{segment}-{feat_extr}-{model}.csv"), index=False)
        summary_scores.to_csv(os.path.join(output_path,f"FINAL_summary_scores-{geneset}-{segment}.csv"), index=False)

        if calibration:
            if verbose: print("saving calibrated results")
            calibration.to_csv(os.path.join(output_path,f"calibration_params-{geneset}-{segment}-{feat_extr}-{model}.csv"))
            per_gene_scores_cal.to_csv(os.path.join(output_path,f"calibrated_per_gene_scores-{geneset}-{segment}-{feat_extr}-{model}.csv"), index=False)
            summary_scores_cal.to_csv(os.path.join(output_path,f"calibrated_summary_scores-{geneset}-{segment}.csv"), index=False)
            

    return y_pred, per_gene_scores, y_pred_cal, per_gene_scores_cal

# Data Load

In [ ]:
root_path = "../data"

In [ ]:
output_path=os.path.join(root_path,'04-results_test')
feature_path=os.path.join(root_path,'02-input','features')
outcome_path=os.path.join(root_path,'02-input','outcomes')
results_path = os.path.join(root_path,'03-results_cv')
summary_path = os.path.join(root_path,'03-results_cv','summary')


In [ ]:
curr_date=str(datetime.date.today())
curr_time=datetime.datetime.now().strftime('%H%M')
name=f"retrain_final_models_{curr_date}_{curr_time}"
print(name)

In [ ]:
#set up paths for models, training curves and predictions
save_path = os.path.join(output_path,name)
os.makedirs(save_path,exist_ok=True)

print(f"ip: {feature_path}")
print(f"op: {save_path}")

Read in params for each gene

In [ ]:
all_gene_scores = pd.read_csv(os.path.join(results_path,'allgenescores.csv'))
all_gene_scores.shape

In [ ]:
all_gene_scores.head()

In [ ]:
def load_data(geneset, feat_extr, segment, test=False, verbose=False):
    if test:
        suffix='_TEST'
    else:
        suffix=''
    if verbose: print(geneset)
    if verbose: print(feat_extr)
    feat_file= f'{feat_extr}-{segment}_features.csv'
    if verbose: print(feat_file)
    truth_file=f'{geneset.lower()}_{segment[:-1].lower()}_raw{suffix}.csv'
    if verbose: print(truth_file)

    # 1. read in outcomes
    #outcome_df = pd.read_csv(os.path.join(truth_path,'forwriteupv1',truth_file))
    outcome_df = pd.read_csv(os.path.join(outcome_path,truth_file))
    outcome_df = outcome_df.set_index('ROI_ID')
    ## doing this to check later if the first gene is NA
    sig_name = outcome_df.columns[0]
    
    if verbose: print(f"feature file: {feat_file}")
    image_names, feature_matrix = load_features_from_csv(os.path.join(feature_path,feat_file))

    """if i'm reading features from a file then have to get metadata from image names"""

    patch_metadata = pd.DataFrame([x.replace('.png','').split('_') for x in image_names],columns=['name', 'LEAP_ID', 'Segment','ROI_num', 'x', 'y'])
    patch_metadata = patch_metadata.set_index('name')
    patch_metadata = patch_metadata.filter(items=['LEAP_ID'])
    # remove patches that we don't have both features and outcomes

    merged_df = patch_metadata.join(outcome_df, how='left')

    mask = ~merged_df[sig_name].isna()
    feature_matrix_clean = feature_matrix[mask.values]
    merged_df = merged_df.dropna(subset=[sig_name])

    return feature_matrix_clean, merged_df


Save gene names

In [ ]:
import joblib
joblib.dump(
    list(set(all_gene_scores.loc[all_gene_scores['Segment']=='CD45+','Gene'].values)),
    os.path.join(root_path,'06-treatment_response','cd45_genelist_full.joblib')
    )
joblib.dump(
    list(set(all_gene_scores.loc[all_gene_scores['Segment']=='PanCK+','Gene'].values)),
    os.path.join(root_path,'06-treatment_response','panck_genelist_full.joblib')
    )

# Train and Evaluate the final models

read in the top models that we need to use

In [ ]:
best_per_segment = pd.read_csv(os.path.join(summary_path,'best_model_per_segment_goodpredictors.csv'))
best_per_segment

In [ ]:
orig_best_per_segment = best_per_segment.copy()

In [ ]:
best_per_segment = best_per_segment[best_per_segment['Feature_Extractor']=='fused']

In [ ]:
all_gene_scores.Feature_Extractor.unique()

In [ ]:
cond = all_gene_scores['GeneSet']=='Panel'
cond = cond & (all_gene_scores['Segment']=='CD45+')
cond = cond & (all_gene_scores['Feature_Extractor']=='fused')
cond = cond & (all_gene_scores['ML_Model']=='SVM')
params_subset = all_gene_scores.loc[cond,['Gene','best_params']]


In [ ]:
contamination_genes = ['IGHG3',
 'IGHG1',
 'IGHG4',
 'IGHG2',
 'IGKC',
 'IGLL5',
 'IGHM',
 'IGHA1',
 'JCHAIN',
 'MZB1']

remove any tumour rows that have the contamination genes

In [ ]:
cond = (all_gene_scores['Segment']=='PanCK+')
cond = cond & (all_gene_scores['Gene'].isin(contamination_genes))
all_gene_scores = all_gene_scores[~cond]

In [ ]:
all_gene_scores[cond]

In [ ]:
from typing import Tuple

In [ ]:
Key = Tuple[str, str, str]  # (feat_extr, segment, geneset)

In [ ]:
def split_leap_id(y: pd.DataFrame) -> Tuple[pd.Series, pd.DataFrame]:
    """Return LEAP_ID series + y without LEAP_ID."""
    if "LEAP_ID" not in y.columns:
        raise ValueError("Expected 'LEAP_ID' column in y.")
    leapids = y["LEAP_ID"]
    return leapids, y.drop(columns=["LEAP_ID"])

In [ ]:
def get_params_subset(
    params_index: pd.DataFrame,
    geneset: str,
    segment: str,
    feat_extr: str,
    model_name: str,
    gene_order: List[str],
    verbose = False
) -> pd.DataFrame:
    """
    Fetch best_params for the run and align them to gene_order.
    Raises if any genes are missing.
    """
    cond = (
        (params_index["GeneSet"] == geneset)
        & (params_index["Segment"] == segment)
        & (params_index["Feature_Extractor"] == feat_extr)
        & (params_index["ML_Model"] == model_name)
    )

    subset = params_index.loc[cond, ["Gene", "best_params"]]

    # Align order to training gene order
    if not np.array_equal(subset["Gene"].to_numpy(), np.array(gene_order, dtype=object)):
        if verbose: print("Sorting params")
        subset = subset.set_index("Gene").reindex(gene_order).reset_index()

    # Fail loudly if reindex introduced missing params
    if subset["best_params"].isna().any():
        missing = subset.loc[subset["best_params"].isna(), "Gene"].tolist()
        raise ValueError(f"Missing best_params for genes: {missing}")

    return subset

In [ ]:
from sklearn.model_selection import GroupShuffleSplit
import numpy as np
import pandas as pd
from gene_model import Calibration

def fit_and_calibrate(
    model_name: str,
    X_train,
    y_train: pd.DataFrame,          # genes only (NO LEAP_ID)
    leapids: pd.Series,             # grouping variable aligned with rows
    params_subset: pd.DataFrame,    # columns: ["Gene", "best_params"] aligned to y_train.columns
    calib_frac: float = 0.2,
    random_state: int = 77,
    verbose=False
):
    """
    Train per-gene models on a group-aware split and estimate affine calibration
    on a held-out calibration split where entire LEAP_ID groups are held out.
    """
    if len(y_train) != len(leapids):
        raise ValueError("y_train and leapids must have the same number of rows.")
    if "LEAP_ID" in y_train.columns:
        raise ValueError("y_train must not contain LEAP_ID; pass leapids separately.")

    # Ensure params are aligned to y_train column order
    if not params_subset["Gene"].tolist() == y_train.columns.tolist():
        raise ValueError("params_subset Gene order must match y_train.columns order.")

    genes = params_subset["Gene"].tolist()
    params = params_subset["best_params"].tolist()

    splitter = GroupShuffleSplit(n_splits=1, test_size=calib_frac, random_state=random_state)
    idx_fit, idx_cal = next(splitter.split(X_train, y_train, groups=leapids))

    # Slice X (works for numpy arrays or pandas DataFrames)
    X_fit = X_train.iloc[idx_fit] if hasattr(X_train, "iloc") else X_train[idx_fit]
    X_cal = X_train.iloc[idx_cal] if hasattr(X_train, "iloc") else X_train[idx_cal]

    y_fit = y_train.iloc[idx_fit]
    y_cal = y_train.iloc[idx_cal]

    # Sanity: no group overlap
    if set(leapids.iloc[idx_fit]).intersection(set(leapids.iloc[idx_cal])):
        raise RuntimeError("Group leakage detected: some LEAP_IDs appear in both fit and calibration sets.")

    trained_models = train_final_regressor(model_name, X_fit, y_fit, genes, params)

    pred_cal = np.column_stack([m.predict(X_cal) for m in trained_models])

    y_cal_np = y_cal.to_numpy()

    ## SANITY CHECK
    sigma_y = y_cal_np.std(axis=0)
    sigma_hat = pred_cal.std(axis=0)

    bad_y = np.where(sigma_y < 1e-12)[0]
    bad_hat = np.where(sigma_hat < 1e-12)[0]

    if len(bad_y):
        print("Genes with ~constant y_cal:", y_train.columns[bad_y].tolist()[:10], "...")
    if len(bad_hat):
        print("Genes with ~constant pred_cal:", y_train.columns[bad_hat].tolist()[:10], "...")




    y_cal_np = y_cal.to_numpy()
    num_genes = y_cal_np.shape[1]
    alphas = np.empty(num_genes, dtype=float)
    betas = np.empty(num_genes, dtype=float)

    for j in range(num_genes):
        alphas[j], betas[j] = affine_calibration(y_cal_np[:, j], pred_cal[:, j])

    # Optional: training predictions for diagnostics only (e.g., variance_ratio)
    pred_fit = np.column_stack([m.predict(X_fit) for m in trained_models])
    calib = Calibration(alphas=alphas, betas=betas,genes=y_train.columns.tolist())

    return trained_models, calib, pred_fit, idx_fit, idx_cal


In [ ]:
def preds_to_df(preds: np.ndarray, y_template: pd.DataFrame, leapids: pd.Series) -> pd.DataFrame:
    """Create a predictions df with same index/columns as y_template and LEAP_ID inserted."""
    df = pd.DataFrame(preds, columns=y_template.columns, index=y_template.index)
    df.insert(0, "LEAP_ID", leapids)
    return df

In [ ]:
def print_duration(from_time,to_time,label="", in_mins=True):
    dur = to_time - from_time
    if in_mins:
        print(f"{label}:\t {(dur/60):.1f} mins")
    else:
        print(f"{label}:\t {dur:.1f} secs")

def print_time_since(from_time,label="",in_mins=False):
    to_time = time.perf_counter()
    print_duration(from_time,to_time,label,in_mins)
    #dur = to_time - from_time

    # if in_mins:
        
    #     print(f"{label}:\t {(dur/60):.1f} mins")
    # else:
    #     print(f"{label}:\t {dur:.1f} secs")
    return to_time

## init

In [ ]:
# Pre-slice once for readability/perf
params_index = all_gene_scores[
    ["GeneSet", "Segment", "Feature_Extractor", "ML_Model", "Gene", "best_params"]
].copy()

seg_test_gene_scores_list = []
seg_test_calib_scores_list = []

final_models: Dict[Key, Dict[str, Any]] = {}
final_calibration: Dict[Key, Dict[str, Any]] = {}
y_train_dict: Dict[Key, pd.DataFrame] = {}
y_test_dict: Dict[Key, pd.DataFrame] = {}
y_pred_dict: Dict[Key, pd.DataFrame] = {}
y_pred_calib_dict: Dict[Key, pd.DataFrame] = {}
train_pred_dict: Dict[Key, pd.DataFrame] = {}

once per row

## CD45

In [ ]:
row = best_per_segment.iloc[0]
segment = row["Segment"]
feat_extr = row["Feature_Extractor"]
model_name = row["ML_Model"]

## CD45 Panel

In [ ]:
geneset = geneset_order[0]

In [ ]:
iter_start = time.perf_counter()

key: Key = (feat_extr, segment, geneset)
print(f"****** {geneset} | {segment} | {feat_extr} | {model_name} ******")

#### -------------------- TRAIN -------------------- ####
X_train, y_train = load_data(geneset, feat_extr, segment)
y_train_dict[key] = y_train

t1 = print_time_since(iter_start,"loading",in_mins=False)

leapids_train, y_train_noleap = split_leap_id(y_train)
gene_order = y_train_noleap.columns.tolist()

params_subset = get_params_subset(
    params_index=params_index,
    geneset=geneset,
    segment=segment,
    feat_extr=feat_extr,
    model_name=model_name,
    gene_order=gene_order,
)

######### TRAIN MODELS ###########################################
#print("TRAIN")
trained_models, calib, train_preds, idx_fit, idx_cal = fit_and_calibrate(
    model_name=model_name,
    X_train=X_train,
    y_train=y_train_noleap,
    leapids=leapids_train,
    params_subset=params_subset,
    calib_frac=CALIBRATION_FRACTION
)

train_pred_dict[key] = preds_to_df(
    train_preds,
    y_train_noleap.iloc[idx_fit],
    leapids_train.iloc[idx_fit],
)

t2 = print_time_since(t1,"training",in_mins=True)


In [ ]:
#### -------------------- TEST -------------------- ####
t2 = time.perf_counter() #print_time_since(t1,"training",in_mins=False)
print("*** TEST ***")
X_test, y_test = load_data(geneset, feat_extr, segment, test=True)
y_test_dict[key] = y_test

leapids_test, y_test_noleap = split_leap_id(y_test)

# Ensure gene order matches between train and test (prevents silent chaos)
if not y_test_noleap.columns.equals(y_train_noleap.columns):
    raise ValueError(
        f"Train/Test gene columns mismatch for key={key}.\n"
        f"Train cols (first 5): {y_train_noleap.columns[:5].tolist()}\n"
        f"Test  cols (first 5): {y_test_noleap.columns[:5].tolist()}"
    )
t3 = print_time_since(t2,"testing",in_mins=False)

In [ ]:
#### -------------------- EVAL -------------------- ####

t3 = time.perf_counter()
y_pred, per_gene_scores, y_pred_calib, per_gene_scores_calib = evaluate_final_models(
    trained_models,
    X_test,
    y_test_noleap,
    geneset,
    segment,
    feat_extr,
    model_name,
    output_path=save_path,
    calibration=calib
)

seg_test_gene_scores_list.append(per_gene_scores)
seg_test_calib_scores_list.append(per_gene_scores_calib)

y_pred_dict[key] = preds_to_df(y_pred, y_test_noleap, leapids_test)
y_pred_calib_dict[key] = preds_to_df(y_pred_calib, y_test_noleap, leapids_test)


# Store final models per gene
final_models[key] = dict(zip(y_test_noleap.columns.tolist(), trained_models))
final_calibration[key] = calib #dict(zip(y_test_noleap.columns.tolist(), calib))

t4 = print_time_since(t3,"eval",in_mins=False)

## CD45 HVG

In [ ]:
geneset = geneset_order[1]

In [ ]:
iter_start = time.perf_counter()

key: Key = (feat_extr, segment, geneset)
print(f"****** {geneset} | {segment} | {feat_extr} | {model_name} ******")

#### -------------------- TRAIN -------------------- ####
X_train, y_train = load_data(geneset, feat_extr, segment)
y_train_dict[key] = y_train

t1 = print_time_since(iter_start,"loading",in_mins=False)

leapids_train, y_train_noleap = split_leap_id(y_train)
gene_order = y_train_noleap.columns.tolist()

params_subset = get_params_subset(
    params_index=params_index,
    geneset=geneset,
    segment=segment,
    feat_extr=feat_extr,
    model_name=model_name,
    gene_order=gene_order,
)

######### TRAIN MODELS ###########################################
#print("TRAIN")
trained_models, calib, train_preds, idx_fit, idx_cal = fit_and_calibrate(
    model_name=model_name,
    X_train=X_train,
    y_train=y_train_noleap,
    leapids=leapids_train,
    params_subset=params_subset,
    calib_frac=CALIBRATION_FRACTION
)

train_pred_dict[key] = preds_to_df(
    train_preds,
    y_train_noleap.iloc[idx_fit],
    leapids_train.iloc[idx_fit],
)

t2 = print_time_since(t1,"training",in_mins=True)


In [ ]:
#### -------------------- TEST -------------------- ####

t2 = time.perf_counter() #print_time_since(t1,"training",in_mins=False)
print("*** TEST ***")
X_test, y_test = load_data(geneset, feat_extr, segment, test=True)
y_test_dict[key] = y_test

leapids_test, y_test_noleap = split_leap_id(y_test)

# Ensure gene order matches between train and test (prevents silent chaos)
if not y_test_noleap.columns.equals(y_train_noleap.columns):
    raise ValueError(
        f"Train/Test gene columns mismatch for key={key}.\n"
        f"Train cols (first 5): {y_train_noleap.columns[:5].tolist()}\n"
        f"Test  cols (first 5): {y_test_noleap.columns[:5].tolist()}"
    )
t3 = print_time_since(t2,"testing",in_mins=False)

In [ ]:
#### -------------------- EVAL -------------------- ####

t3 = time.perf_counter()
y_pred, per_gene_scores, y_pred_calib, per_gene_scores_calib = evaluate_final_models(
    trained_models,
    X_test,
    y_test_noleap,
    geneset,
    segment,
    feat_extr,
    model_name,
    output_path=save_path,
    calibration=calib
)

seg_test_gene_scores_list.append(per_gene_scores)
seg_test_calib_scores_list.append(per_gene_scores_calib)

y_pred_dict[key] = preds_to_df(y_pred, y_test_noleap, leapids_test)
y_pred_calib_dict[key] = preds_to_df(y_pred_calib, y_test_noleap, leapids_test)


# Store final models per gene
final_models[key] = dict(zip(y_test_noleap.columns.tolist(), trained_models))
final_calibration[key] = calib #dict(zip(y_test_noleap.columns.tolist(), calib))

t4 = print_time_since(t3,"eval",in_mins=False)

## CD45 SVG

In [ ]:
geneset = geneset_order[2]

In [ ]:
iter_start = time.perf_counter()

key: Key = (feat_extr, segment, geneset)
print(f"****** {geneset} | {segment} | {feat_extr} | {model_name} ******")

#### -------------------- TRAIN -------------------- ####
X_train, y_train = load_data(geneset, feat_extr, segment)
y_train_dict[key] = y_train

t1 = print_time_since(iter_start,"loading",in_mins=False)

leapids_train, y_train_noleap = split_leap_id(y_train)
gene_order = y_train_noleap.columns.tolist()

params_subset = get_params_subset(
    params_index=params_index,
    geneset=geneset,
    segment=segment,
    feat_extr=feat_extr,
    model_name=model_name,
    gene_order=gene_order,
)

######### TRAIN MODELS ###########################################
#print("TRAIN")
trained_models, calib, train_preds, idx_fit, idx_cal = fit_and_calibrate(
    model_name=model_name,
    X_train=X_train,
    y_train=y_train_noleap,
    leapids=leapids_train,
    params_subset=params_subset,
    calib_frac=CALIBRATION_FRACTION
)

train_pred_dict[key] = preds_to_df(
    train_preds,
    y_train_noleap.iloc[idx_fit],
    leapids_train.iloc[idx_fit],
)

t2 = print_time_since(t1,"training",in_mins=True)


In [ ]:
#### -------------------- TEST -------------------- ####

t2 = time.perf_counter() #print_time_since(t1,"training",in_mins=False)
print("*** TEST ***")
X_test, y_test = load_data(geneset, feat_extr, segment, test=True)
y_test_dict[key] = y_test

leapids_test, y_test_noleap = split_leap_id(y_test)

# Ensure gene order matches between train and test (prevents silent chaos)
if not y_test_noleap.columns.equals(y_train_noleap.columns):
    raise ValueError(
        f"Train/Test gene columns mismatch for key={key}.\n"
        f"Train cols (first 5): {y_train_noleap.columns[:5].tolist()}\n"
        f"Test  cols (first 5): {y_test_noleap.columns[:5].tolist()}"
    )
t3 = print_time_since(t2,"testing",in_mins=False)

In [ ]:
#### -------------------- EVAL -------------------- ####

t3 = time.perf_counter()
y_pred, per_gene_scores, y_pred_calib, per_gene_scores_calib = evaluate_final_models(
    trained_models,
    X_test,
    y_test_noleap,
    geneset,
    segment,
    feat_extr,
    model_name,
    output_path=save_path,
    calibration=calib
)

seg_test_gene_scores_list.append(per_gene_scores)
seg_test_calib_scores_list.append(per_gene_scores_calib)

y_pred_dict[key] = preds_to_df(y_pred, y_test_noleap, leapids_test)
y_pred_calib_dict[key] = preds_to_df(y_pred_calib, y_test_noleap, leapids_test)


# Store final models per gene
final_models[key] = dict(zip(y_test_noleap.columns.tolist(), trained_models))
final_calibration[key] = calib #dict(zip(y_test_noleap.columns.tolist(), calib))

t4 = print_time_since(t3,"eval",in_mins=False)

## PanCK

In [ ]:
row = best_per_segment.iloc[1]
segment = row["Segment"]
feat_extr = row["Feature_Extractor"]
model_name = row["ML_Model"]

## PanCK Panel

In [ ]:
geneset = geneset_order[0]

In [ ]:
iter_start = time.perf_counter()

key: Key = (feat_extr, segment, geneset)
print(f"****** {geneset} | {segment} | {feat_extr} | {model_name} ******")

#### -------------------- TRAIN -------------------- ####
X_train, y_train = load_data(geneset, feat_extr, segment)
y_train_dict[key] = y_train

t1 = print_time_since(iter_start,"loading",in_mins=False)

leapids_train, y_train_noleap = split_leap_id(y_train)
gene_order = y_train_noleap.columns.tolist()

params_subset = get_params_subset(
    params_index=params_index,
    geneset=geneset,
    segment=segment,
    feat_extr=feat_extr,
    model_name=model_name,
    gene_order=gene_order,
)

######### TRAIN MODELS ###########################################
#print("TRAIN")
trained_models, calib, train_preds, idx_fit, idx_cal = fit_and_calibrate(
    model_name=model_name,
    X_train=X_train,
    y_train=y_train_noleap,
    leapids=leapids_train,
    params_subset=params_subset,
    calib_frac=CALIBRATION_FRACTION
)

train_pred_dict[key] = preds_to_df(
    train_preds,
    y_train_noleap.iloc[idx_fit],
    leapids_train.iloc[idx_fit],
)

t2 = print_time_since(t1,"training",in_mins=True)


In [ ]:
#### -------------------- TEST -------------------- ####
t2 = time.perf_counter() #print_time_since(t1,"training",in_mins=False)
print("*** TEST ***")
X_test, y_test = load_data(geneset, feat_extr, segment, test=True)
y_test_dict[key] = y_test

leapids_test, y_test_noleap = split_leap_id(y_test)

# Ensure gene order matches between train and test (prevents silent chaos)
if not y_test_noleap.columns.equals(y_train_noleap.columns):
    raise ValueError(
        f"Train/Test gene columns mismatch for key={key}.\n"
        f"Train cols (first 5): {y_train_noleap.columns[:5].tolist()}\n"
        f"Test  cols (first 5): {y_test_noleap.columns[:5].tolist()}"
    )
t3 = print_time_since(t2,"testing",in_mins=False)

In [ ]:
#### -------------------- EVAL -------------------- ####

t3 = time.perf_counter()
y_pred, per_gene_scores, y_pred_calib, per_gene_scores_calib = evaluate_final_models(
    trained_models,
    X_test,
    y_test_noleap,
    geneset,
    segment,
    feat_extr,
    model_name,
    output_path=save_path,
    calibration=calib
)

seg_test_gene_scores_list.append(per_gene_scores)
seg_test_calib_scores_list.append(per_gene_scores_calib)

y_pred_dict[key] = preds_to_df(y_pred, y_test_noleap, leapids_test)
y_pred_calib_dict[key] = preds_to_df(y_pred_calib, y_test_noleap, leapids_test)


# Store final models per gene
final_models[key] = dict(zip(y_test_noleap.columns.tolist(), trained_models))
final_calibration[key] = calib #dict(zip(y_test_noleap.columns.tolist(), calib))

t4 = print_time_since(t3,"eval",in_mins=False)

## PanCK HVG

In [ ]:
geneset = geneset_order[1]

In [ ]:
iter_start = time.perf_counter()

key: Key = (feat_extr, segment, geneset)
print(f"****** {geneset} | {segment} | {feat_extr} | {model_name} ******")

#### -------------------- TRAIN -------------------- ####
X_train, y_train = load_data(geneset, feat_extr, segment)
y_train_dict[key] = y_train

t1 = print_time_since(iter_start,"loading",in_mins=False)

leapids_train, y_train_noleap = split_leap_id(y_train)
gene_order = y_train_noleap.columns.tolist()

params_subset = get_params_subset(
    params_index=params_index,
    geneset=geneset,
    segment=segment,
    feat_extr=feat_extr,
    model_name=model_name,
    gene_order=gene_order,
)

######### TRAIN MODELS ###########################################
#print("TRAIN")
trained_models, calib, train_preds, idx_fit, idx_cal = fit_and_calibrate(
    model_name=model_name,
    X_train=X_train,
    y_train=y_train_noleap,
    leapids=leapids_train,
    params_subset=params_subset,
    calib_frac=CALIBRATION_FRACTION
)

train_pred_dict[key] = preds_to_df(
    train_preds,
    y_train_noleap.iloc[idx_fit],
    leapids_train.iloc[idx_fit],
)

t2 = print_time_since(t1,"training",in_mins=True)


In [ ]:
#### -------------------- TEST -------------------- ####
t2 = time.perf_counter() #print_time_since(t1,"training",in_mins=False)
print("*** TEST ***")
X_test, y_test = load_data(geneset, feat_extr, segment, test=True)
y_test_dict[key] = y_test

leapids_test, y_test_noleap = split_leap_id(y_test)

# Ensure gene order matches between train and test (prevents silent chaos)
if not y_test_noleap.columns.equals(y_train_noleap.columns):
    raise ValueError(
        f"Train/Test gene columns mismatch for key={key}.\n"
        f"Train cols (first 5): {y_train_noleap.columns[:5].tolist()}\n"
        f"Test  cols (first 5): {y_test_noleap.columns[:5].tolist()}"
    )
t3 = print_time_since(t2,"testing",in_mins=False)

In [ ]:
#### -------------------- EVAL -------------------- ####

t3 = time.perf_counter()
y_pred, per_gene_scores, y_pred_calib, per_gene_scores_calib = evaluate_final_models(
    trained_models,
    X_test,
    y_test_noleap,
    geneset,
    segment,
    feat_extr,
    model_name,
    output_path=save_path,
    calibration=calib
)

seg_test_gene_scores_list.append(per_gene_scores)
seg_test_calib_scores_list.append(per_gene_scores_calib)

y_pred_dict[key] = preds_to_df(y_pred, y_test_noleap, leapids_test)
y_pred_calib_dict[key] = preds_to_df(y_pred_calib, y_test_noleap, leapids_test)


# Store final models per gene
final_models[key] = dict(zip(y_test_noleap.columns.tolist(), trained_models))
final_calibration[key] = calib #dict(zip(y_test_noleap.columns.tolist(), calib))

t4 = print_time_since(t3,"eval",in_mins=False)

## PanCK SVG

In [ ]:
geneset = geneset_order[2]

In [ ]:
iter_start = time.perf_counter()

key: Key = (feat_extr, segment, geneset)
print(f"****** {geneset} | {segment} | {feat_extr} | {model_name} ******")

#### -------------------- TRAIN -------------------- ####
X_train, y_train = load_data(geneset, feat_extr, segment)
y_train_dict[key] = y_train

t1 = print_time_since(iter_start,"loading",in_mins=False)

leapids_train, y_train_noleap = split_leap_id(y_train)
gene_order = y_train_noleap.columns.tolist()

params_subset = get_params_subset(
    params_index=params_index,
    geneset=geneset,
    segment=segment,
    feat_extr=feat_extr,
    model_name=model_name,
    gene_order=gene_order,
)

######### TRAIN MODELS ###########################################
#print("TRAIN")
trained_models, calib, train_preds, idx_fit, idx_cal = fit_and_calibrate(
    model_name=model_name,
    X_train=X_train,
    y_train=y_train_noleap,
    leapids=leapids_train,
    params_subset=params_subset,
    calib_frac=CALIBRATION_FRACTION
)

train_pred_dict[key] = preds_to_df(
    train_preds,
    y_train_noleap.iloc[idx_fit],
    leapids_train.iloc[idx_fit],
)

t2 = print_time_since(t1,"training",in_mins=True)


In [ ]:
#### -------------------- TEST -------------------- ####
t2 = time.perf_counter() #print_time_since(t1,"training",in_mins=False)
print("*** TEST ***")
X_test, y_test = load_data(geneset, feat_extr, segment, test=True)
y_test_dict[key] = y_test

leapids_test, y_test_noleap = split_leap_id(y_test)

# Ensure gene order matches between train and test (prevents silent chaos)
if not y_test_noleap.columns.equals(y_train_noleap.columns):
    raise ValueError(
        f"Train/Test gene columns mismatch for key={key}.\n"
        f"Train cols (first 5): {y_train_noleap.columns[:5].tolist()}\n"
        f"Test  cols (first 5): {y_test_noleap.columns[:5].tolist()}"
    )
t3 = print_time_since(t2,"testing",in_mins=False)

In [ ]:
#### -------------------- EVAL -------------------- ####

t3 = time.perf_counter()
y_pred, per_gene_scores, y_pred_calib, per_gene_scores_calib = evaluate_final_models(
    trained_models,
    X_test,
    y_test_noleap,
    geneset,
    segment,
    feat_extr,
    model_name,
    output_path=save_path,
    calibration=calib
)

seg_test_gene_scores_list.append(per_gene_scores)
seg_test_calib_scores_list.append(per_gene_scores_calib)

y_pred_dict[key] = preds_to_df(y_pred, y_test_noleap, leapids_test)
y_pred_calib_dict[key] = preds_to_df(y_pred_calib, y_test_noleap, leapids_test)


# Store final models per gene
final_models[key] = dict(zip(y_test_noleap.columns.tolist(), trained_models))
final_calibration[key] = calib #dict(zip(y_test_noleap.columns.tolist(), calib))

t4 = print_time_since(t3,"eval",in_mins=False)

In [ ]:
t5 = print_time_since(start_time,"time since starting script")

## THIS IS THE MAIN LOOP THAT SHOULD IDEALLY BE USED
Had to separate it due to memory constraints

In [ ]:
# ------------------------------- main -------------------------------- #



# # Pre-slice once for readability/perf
# params_index = all_gene_scores[
#     ["GeneSet", "Segment", "Feature_Extractor", "ML_Model", "Gene", "best_params"]
# ].copy()

# seg_test_gene_scores_list = []
# seg_test_calib_scores_list = []

# final_models: Dict[Key, Dict[str, Any]] = {}
# final_calibration: Dict[Key, Dict[str, Any]] = {}
# y_train_dict: Dict[Key, pd.DataFrame] = {}
# y_test_dict: Dict[Key, pd.DataFrame] = {}
# y_pred_dict: Dict[Key, pd.DataFrame] = {}
# y_pred_calib_dict: Dict[Key, pd.DataFrame] = {}
# train_pred_dict: Dict[Key, pd.DataFrame] = {}

# for _, row in best_per_segment.iterrows():
#     segment = row["Segment"]
#     feat_extr = row["Feature_Extractor"]
#     model_name = row["ML_Model"]

#     for geneset in geneset_order:
#         key: Key = (feat_extr, segment, geneset)
#         print(f"****** {geneset} | {segment} | {feat_extr} | {model_name} ******")

#         # -------------------- TRAIN -------------------- #
#         X_train, y_train = load_data(geneset, feat_extr, segment)
#         y_train_dict[key] = y_train

        
#         print("Feature Matrix Shape:", X_train.shape)
#         print("Outcome Shape:", len(y_train))

#         leapids_train, y_train_noleap = split_leap_id(y_train)
#         gene_order = y_train_noleap.columns.tolist()

#         params_subset = get_params_subset(
#             params_index=params_index,
#             geneset=geneset,
#             segment=segment,
#             feat_extr=feat_extr,
#             model_name=model_name,
#             gene_order=gene_order,
#         )

        
#         ######### TRAIN MODELS ###########################################
#         print("TRAIN")
#         trained_models, calib, train_preds, idx_fit, idx_cal = fit_and_calibrate(
#             model_name=model_name,
#             X_train=X_train,
#             y_train=y_train_noleap,
#             leapids=leapids_train,
#             params_subset=params_subset,
#             calib_frac=CALIBRATION_FRACTION
#         )

#         train_pred_dict[key] = preds_to_df(
#             train_preds,
#             y_train_noleap.iloc[idx_fit],
#             leapids_train.iloc[idx_fit],
#         )
#         # sanity metric
#         print("var ratio", variance_ratio(y_train_noleap.iloc[idx_fit], train_preds))


#         ##################################################################################
#         ######## EVALUATE ON HOLD OUT TEST SET ###########################################
#         ##################################################################################
#         -------------------- TEST -------------------- #
#         print("*** TEST ***")
#         X_test, y_test = load_data(geneset, feat_extr, segment, test=True)
#         y_test_dict[key] = y_test

#         print("Feature Matrix Shape:", X_test.shape)
#         print("Outcome Shape:", len(y_test))

#         leapids_test, y_test_noleap = split_leap_id(y_test)

#         # Ensure gene order matches between train and test (prevents silent chaos)
#         if not y_test_noleap.columns.equals(y_train_noleap.columns):
#             raise ValueError(
#                 f"Train/Test gene columns mismatch for key={key}.\n"
#                 f"Train cols (first 5): {y_train_noleap.columns[:5].tolist()}\n"
#                 f"Test  cols (first 5): {y_test_noleap.columns[:5].tolist()}"
#             )

#         #### EVALUATE ######################################################################
#         y_pred, per_gene_scores, y_pred_calib, per_gene_scores_calib = evaluate_final_models(
#             trained_models,
#             X_test,
#             y_test_noleap,
#             geneset,
#             segment,
#             feat_extr,
#             model_name,
#             output_path=save_path,
#             calibration=calib
#         )

#         seg_test_gene_scores_list.append(per_gene_scores)
#         seg_test_calib_scores_list.append(per_gene_scores_calib)

#         y_pred_dict[key] = preds_to_df(y_pred, y_test_noleap, leapids_test)
#         y_pred_calib_dict[key] = preds_to_df(y_pred_calib, y_test_noleap, leapids_test)

        
#         # Store final models per gene
#         final_models[key] = dict(zip(y_test_noleap.columns.tolist(), trained_models))
#         final_calibration[key] = calib #dict(zip(y_test_noleap.columns.tolist(), calib))

### Save models

In [ ]:
import joblib
import json
def save_gene_models(model_dict, save_path):
    os.makedirs(save_path, exist_ok=True)

    for gene, model in model_dict.items():
        joblib.dump(model, f"{save_path}/{gene}.joblib")

    with open(f"{save_path}/index.json", "w") as f:
        json.dump(list(model_dict.keys()), f, indent=2)

In [ ]:
model_save_path = os.path.join(root_path, "models",name)
os.makedirs(model_save_path,exist_ok=True)

In [ ]:
from gene_model import Calibration

def merge_calibs(calib1, calib2, overwrite_dups=True) -> "Calibration":
    if set(calib1.genes) & set(calib2.genes):
        if overwrite_dups:
            print("warning, overlapping genes in merge sets - overwriting the original")
        else:
            raise ValueError("Overlapping genes in calibration merge.")

    # ---- Remove overlapping genes from self ----
    keep_mask_self = [g not in calib2.genes for g in calib1.genes]

    filtered_self_genes = [g for g, k in zip(calib1.genes, keep_mask_self) if k]
    filtered_self_alpha = calib1.alphas[keep_mask_self]
    filtered_self_beta  = calib1.betas[keep_mask_self]

    # ---- Combine: filtered self first, then full other ----
    new_genes = filtered_self_genes + calib2.genes
    new_alpha = np.concatenate([filtered_self_alpha, calib2.alphas])
    new_beta  = np.concatenate([filtered_self_beta, calib2.betas])

    return Calibration(
        alphas=new_alpha,
        betas=new_beta,
        genes=new_genes
    )

In [ ]:
calibration_cd45 = (
    final_calibration[('fused','CD45+','HVG')]
    .merge(final_calibration[('fused','CD45+','SVG')])
    .merge(final_calibration[('fused','CD45+','Panel')])
)

In [ ]:
calibration_panck = (
    final_calibration[('fused','PanCK+','HVG')]
    .merge(final_calibration[('fused','PanCK+','SVG')])
    .merge(final_calibration[('fused','PanCK+','Panel')])
)

In [ ]:
models_cd45 = final_models[('fused','CD45+','HVG')] \
                | final_models[('fused','CD45+','SVG')] \
                | final_models[('fused','CD45+','Panel')]

save_gene_models(models_cd45, os.path.join(model_save_path,'trained_models','cd45'))
print(len(models_cd45))

In [ ]:
models_panck = final_models[('fused','PanCK+','HVG')] \
                | final_models[('fused','PanCK+','SVG')] \
                | final_models[('fused','PanCK+','Panel')]
save_gene_models(models_panck, os.path.join(model_save_path,'trained_models','panck'))
print(len(models_panck))

In [ ]:
t6 = print_time_since(t5)
model_save_path

saving takes ~2 mins each

In [ ]:
from gene_model import GeneModelStore


store_cd45 = GeneModelStore(models=models_cd45,
                            calibration=calibration_cd45,
                            model_dir=os.path.join(model_save_path,'trained_models','cd45'))
store_cd45.save_all()
t7 = print_time_since(t6, in_mins=True)

In [ ]:


store_panck = GeneModelStore(models=models_panck,
                            calibration=calibration_panck,
                            model_dir=os.path.join(model_save_path,'trained_models','panck'))
store_panck.save_all()
t8 = print_time_since(t7,in_mins=True)

### FINISHED TRAIN

In [ ]:
pred_path = os.path.join(root_path,'04-results_test','predicted_expression') 
os.makedirs(pred_path, exists_ok=True)

In [ ]:
for segment, fm in zip(best_per_segment.iloc[:,0].to_list(), best_per_segment.iloc[:,1].to_list()):
    print(segment, fm)

    train_pred = train_pred_dict[(fm, segment, 'SVG')]
    train_pred = train_pred.join(train_pred_dict[(fm, segment, 'HVG')], how='left',rsuffix='_drop')
    train_pred = train_pred.join(train_pred_dict[(fm, segment, 'Panel')], how='left',rsuffix='_drop')
    train_pred = train_pred.drop(columns=train_pred.filter(like='_drop').columns)
    train_pred.to_csv(os.path.join(pred_path,f'trainpreds_{segment[:-1]}_{fm}_SVM.csv'))



In [ ]:
backup_train_preds = train_pred_dict.copy

In [ ]:
best_per_segment = orig_best_per_segment.copy()
best_per_segment

move LEAP_ID to the first column

In [ ]:
for k,v in y_pred_dict.items():
    print(k)
    cols = list(v.columns)
    new_order = ['LEAP_ID'] + sorted([c for c in cols if c != 'LEAP_ID'])
    y_pred_dict[k] = v[new_order]

In [ ]:
for k,v in y_pred_calib_dict.items():
    print(k)
    cols = list(v.columns)
    new_order = ['LEAP_ID'] + sorted([c for c in cols if c != 'LEAP_ID'])
    y_pred_calib_dict[k] = v[new_order]

In [ ]:
y_pred_comb_dict = {}
y_pred_calib_comb_dict = {}

In [ ]:
feat_extr='fused'
for segment in ['CD45+','PanCK+']:
    for geneset in geneset_order:
        print(f"{segment}\t{feat_extr}\t{geneset}")
        pred_cols = y_test_dict[(feat_extr,segment, geneset)].columns[1:].tolist()

        #print(pred_cols)
        df = pd.DataFrame(y_pred_dict[(feat_extr,segment, geneset)],columns=pred_cols)
        df.index = y_test_dict[(feat_extr,segment, geneset)].index
        df.insert(0, 'LEAP_ID', y_test_dict[(feat_extr,segment, geneset)]['LEAP_ID'])
        y_pred_comb_dict[(feat_extr,segment, geneset)] = df

        
        df = pd.DataFrame(y_pred_calib_dict[(feat_extr,segment, geneset)],columns=pred_cols)
        df.index = y_test_dict[(feat_extr,segment, geneset)].index
        df.insert(0, 'LEAP_ID', y_test_dict[(feat_extr,segment, geneset)]['LEAP_ID'])
        y_pred_calib_comb_dict[(feat_extr,segment, geneset)] = df

Save predictions for combined features (unique genes per segment)

In [ ]:
unique_genes={}
for segment in ['CD45+','PanCK+']:
    print(segment, feat_extr)
    y_pred_seg = y_pred_comb_dict[(feat_extr, segment, 'SVG')]
    y_pred_seg = y_pred_seg.join(y_pred_comb_dict[(feat_extr, segment, 'HVG')], how='left',rsuffix='_drop')
    y_pred_seg = y_pred_seg.join(y_pred_comb_dict[(feat_extr, segment, 'Panel')], how='left',rsuffix='_drop')
    y_pred_seg = y_pred_seg.drop(columns=y_pred_seg.filter(like='_drop').columns)
    unique_genes[segment] = sorted(y_pred_seg.iloc[:,1:].columns.to_list())
    y_pred_seg.to_csv(os.path.join(pred_path,f'preds_{segment[:-1]}_{feat_extr}_SVM.csv'))

    ## calibrated predictions
    
    y_pred_seg = y_pred_calib_comb_dict[(feat_extr, segment, 'SVG')]
    y_pred_seg = y_pred_seg.join(y_pred_calib_comb_dict[(feat_extr, segment, 'HVG')], how='left',rsuffix='_drop')
    y_pred_seg = y_pred_seg.join(y_pred_calib_comb_dict[(feat_extr, segment, 'Panel')], how='left',rsuffix='_drop')
    y_pred_seg = y_pred_seg.drop(columns=y_pred_seg.filter(like='_drop').columns)
    unique_genes[segment] = sorted(y_pred_seg.iloc[:,1:].columns.to_list())
    y_pred_seg.to_csv(os.path.join(pred_path,f'preds_calibrated_{segment[:-1]}_{feat_extr}_SVM.csv'))

In [ ]:
cd45_rois = y_pred_comb_dict[(feat_extr,'CD45+', 'Panel')][['LEAP_ID']]
panck_rois = y_pred_comb_dict[(feat_extr,'PanCK+', 'Panel')][['LEAP_ID']]

In [ ]:
cd45_rois.to_csv(os.path.join(save_path,'cd45_rois.csv'))
panck_rois.to_csv(os.path.join(save_path,'panck_rois.csv'))

In [ ]:
cd45_rois

In [ ]:
seg_test_gene_scores = pd.concat(seg_test_gene_scores_list, ignore_index=True, sort=False)
seg_test_calib_scores = pd.concat(seg_test_calib_scores_list, ignore_index=True, sort=False)

PearsonCorr is NaN because all the predicted values were equal for that gene. This is ok as we will only analyse the good predictors

In [ ]:
seg_test_gene_scores[seg_test_gene_scores['PearsonPVals'].isna()]

In [ ]:
seg_test_gene_scores = seg_test_gene_scores.drop_duplicates(subset=['Segment','Feature_Extractor','Gene'])
seg_test_calib_scores = seg_test_calib_scores.drop_duplicates(subset=['Segment','Feature_Extractor','Gene'])

In [ ]:
seg_test_gene_scores.head()

In [ ]:
seg_test_gene_scores[seg_test_gene_scores['Feature_Extractor']==feat_extr].to_csv(os.path.join(save_path,'fused-svm-genescores.csv'),index=False)
seg_test_calib_scores[seg_test_calib_scores['Feature_Extractor']==feat_extr].to_csv(os.path.join(save_path,'fused-svm-genescores_calibrated.csv'),index=False)

## Merge combined preds

y_test and y_pred for analysis are just the fused features (not Virchow/UNI)

In [ ]:
def merge_unique(y_dict, segment='CD45+',feat_extr='fused'):
    # Step 1: Transpose and set column names as row index
    df_panel = y_dict[(feat_extr,segment, 'Panel')].drop(columns='LEAP_ID').T.rename_axis('Gene')#.reset_index()
    df_hvg = y_dict[(feat_extr,segment, 'HVG')].drop(columns=['LEAP_ID']).T.rename_axis('Gene')#.reset_index()
    df_svg = y_dict[(feat_extr,segment, 'SVG')].drop(columns=['LEAP_ID']).T.rename_axis('Gene')#.reset_index()
    # Step 2: 
    # using the row index
    y_df = pd.concat([df_panel,df_hvg,df_svg]).sort_index()
    y_df = y_df.drop_duplicates()
    return y_df


In [ ]:
y_train_cd45 = merge_unique(y_train_dict,segment='CD45+')
y_trainpred_cd45 = merge_unique(train_pred_dict,segment='CD45+')
y_test_cd45 = merge_unique(y_test_dict,segment='CD45+')
y_pred_cd45 = merge_unique(y_pred_dict,segment='CD45+')

y_train_panck = merge_unique(y_train_dict,segment='PanCK+')
y_trainpred_panck = merge_unique(train_pred_dict,segment='PanCK+')
y_test_panck = merge_unique(y_test_dict,segment='PanCK+')
y_pred_panck = merge_unique(y_pred_dict,segment='PanCK+')

##same for calibrated scores
y_pred_calib_cd45 = merge_unique(y_pred_calib_dict,segment='CD45+')
y_pred_calib_panck = merge_unique(y_pred_calib_dict,segment='PanCK+')

same for calibrated scores

In [ ]:
y_pred_calib_cd45.loc['IGHA1']

In [ ]:
seg_test_gene_scores[seg_test_gene_scores['Gene']=='MGP']

In [ ]:
seg_test_calib_scores[seg_test_calib_scores['Gene']=='MGP']

# Analyse Results

## Assess relationship between accuracy and variance


## Proportion of significant genes 

In [ ]:
signif_thresh = 0.05

In [ ]:
cond = seg_test_gene_scores['Feature_Extractor']=='fused'
seg_test_gene_scores[seg_test_gene_scores['Gene']=='APOD']

In [ ]:
cols_to_add = ['nRMSE', 'R2', 'PearsonCorr', 'PearsonPVals','PearsonQVals', 'PearsonCI_low', 'PearsonCI_high','VarianceRatio', 'ConcordanceCorr']

In [ ]:
combined_scores = seg_test_gene_scores[seg_test_gene_scores['Feature_Extractor']=='fused']
combined_scores_calib = seg_test_calib_scores.loc[seg_test_calib_scores['Feature_Extractor']=='fused',cols_to_add].add_suffix('_calib')

combined_scores_unique = combined_scores.join(
    combined_scores_calib,
    how='left'
)

In [ ]:
combined_scores_unique.shape

In [ ]:
combined_scores_unique.head()

In [ ]:

combined_scores_unique = combined_scores_unique.drop_duplicates(subset=['Segment','Gene'])
print(f"#combined: {combined_scores.shape[0]}\t #unique: {combined_scores_unique.shape[0]}")

In [ ]:
num_signif = combined_scores_unique[combined_scores_unique['PearsonQVals']<signif_thresh].shape[0]
print(f"Percent significant: {(num_signif / combined_scores_unique.shape[0]):.2%}")


Percent significant: 95.43%

In [ ]:
num_signif

209

CD45+ only

In [ ]:
segment = 'CD45+'
cond = (combined_scores_unique['Segment']==segment) & (combined_scores_unique['PearsonQVals']<signif_thresh)
num_signif = combined_scores_unique.loc[cond,:].shape[0]
pct_signif = num_signif / combined_scores_unique[combined_scores_unique['Segment']==segment].shape[0]
print(segment)
print(f"num significant: {(num_signif )}")
print(f"Percent significant: {(pct_signif):.2%}")

PanCK+ onlyl

In [ ]:
segment = 'PanCK+'
cond = (combined_scores_unique['Segment']==segment) & (combined_scores_unique['PearsonQVals']<signif_thresh)
num_signif = combined_scores_unique.loc[cond,:].shape[0]
pct_signif = num_signif / combined_scores_unique[combined_scores_unique['Segment']==segment].shape[0]
print(segment)
print(f"num significant: {(num_signif )}")
print(f"Percent significant: {(pct_signif):.2%}")


check which genes were not significantly correlated

In [ ]:
cond1 = (combined_scores_unique['Segment']==segment) & (combined_scores_unique['PearsonQVals']<signif_thresh)
combined_scores_unique.loc[cond1,:].shape

In [ ]:
cond2 = (combined_scores_unique['Segment']==segment)
combined_scores_unique.loc[cond2,:].shape

check which genes don't meet the threshold for PanCK+

In [ ]:
bad_genes_panck = set(combined_scores_unique.loc[cond2,'Gene'].values).difference(set(combined_scores_unique.loc[cond1,'Gene']))

In [ ]:
combined_scores_unique[cond2&(combined_scores_unique['Gene'].isin(bad_genes_panck))]

In [ ]:
cols_to_show = ['Segment','Gene','PearsonCorr','PearsonCorr_calib','PearsonPVals','PearsonQVals', 'PearsonQVals_calib']
combined_scores_unique.loc[combined_scores_unique['PearsonQVals']>=signif_thresh, cols_to_show]

Save the genes that meet our threshold for a "good" predictor:
- FDR-adjusted p-value < 0.05
- R2 > 0.2

In [ ]:
#geneset_good_models = seg_test_gene_scores[seg_test_gene_scores['R2']>0.2].sort_values(by='nRMSE')#
cond = (combined_scores_unique['R2']>0.2)
cond = cond & (combined_scores_unique['PearsonQVals']<0.05)
#cond = cond & (combined_scores['PearsonCorr']>0.5)
geneset_good_models = combined_scores_unique[cond].sort_values(by='PearsonCorr')
geneset_good_models = geneset_good_models.drop_duplicates(subset=['Segment','Gene'])

In [ ]:
combined_scores_unique.shape

In [ ]:
geneset_good_models.shape

In [ ]:
geneset_good_models.head()

In [ ]:
cond = (geneset_good_models['R2']>0.2)
cond = cond & (geneset_good_models['PearsonQVals']<0.05)
cond = cond & (geneset_good_models['PearsonCorr']>0.5)
print("genes over all threshold")
print("CD45:",geneset_good_models[cond&(geneset_good_models['Segment']=='CD45+')].shape[0])
print("PanCK:",geneset_good_models[cond&(geneset_good_models['Segment']=='PanCK+')].shape[0])

In [ ]:
cond = (seg_test_gene_scores['R2']>0.2)
cond = cond & (seg_test_gene_scores['PearsonQVals']<0.05)
#cond = cond & (seg_test_gene_scores['PearsonCorr']>0.5)
geneset_good_models = geneset_good_models[cond]
geneset_good_models.shape

In [ ]:
gene_segment_list = (
    geneset_good_models['Gene']
    + '_'
    + geneset_good_models['Segment'].str.replace('+', '', regex=False)
).tolist()
gene_segment_list

In [ ]:
import joblib
joblib.dump(gene_segment_list,os.path.join(save_path,'robust_genes.joblib') )

In [ ]:
print("unique genes")
print("CD45:",combined_scores_unique[combined_scores_unique['Segment']=='CD45+'].shape[0])
print("PanCK:",combined_scores_unique[combined_scores_unique['Segment']=='PanCK+'].shape[0])

In [ ]:
cond = (geneset_good_models['R2']>0.4)
print("R2 over 0.4")
print("CD45:",combined_scores_unique[cond&(combined_scores_unique['Segment']=='CD45+')].shape[0])
print("PanCK:",combined_scores_unique[cond&(combined_scores_unique['Segment']=='PanCK+')].shape[0])

Check gene scores

In [ ]:
combined_scores_unique.columns

In [ ]:
cols = ['Segment','Gene','R2','PearsonCorr','PearsonQVals','VarianceRatio', 'ConcordanceCorr',
        'R2_calib','VarianceRatio_calib', 'ConcordanceCorr_calib']

In [ ]:
combined_scores_unique.loc[combined_scores_unique['Gene']=='TP63',cols]

MYC - NOT



In [ ]:
combined_scores_unique.loc[combined_scores_unique['Gene']=='MYC',cols]

BCL2 - good in CD45 not PanCK

In [ ]:
combined_scores_unique.loc[combined_scores_unique['Gene']=='BCL2',cols]

SOX10 

In [ ]:
combined_scores_unique.loc[combined_scores_unique['Gene']=='SOX10',cols]

### PD-L1

In [ ]:
combined_scores_unique.loc[combined_scores_unique['Gene']=='CD274',cols]

### EGFR axis

In [ ]:
combined_scores_unique.loc[combined_scores_unique['Gene'].isin(['AREG','EGFR']),cols]

### Chemokines

In [ ]:
combined_scores_unique.loc[combined_scores_unique['Gene'].isin(['CXCL9','CXCL1']),cols]

### TILS

everything above R2 0.5 and PCC 0.7

In [ ]:
combined_scores_unique.loc[combined_scores_unique['Gene'].isin(['CD8A','CD4','CD3D','STAT1','CD163','IGKC','IGHG1','IGHG2','IGHG3','IGHG4']),cols]

### ENO1 

In [ ]:
combined_scores_unique.loc[combined_scores_unique['Gene'].isin(['ENO1']),cols]

Save to CSV

In [ ]:
genes_of_interest=['TP63','MYC','BCL2','SOX10','CD274','AREG','EGFR',
'ENO1',
'CXCL9','CXCL1','CD8A','CD4','CD3D','STAT1','CD163','IGKC','IGHG1','IGHG2','IGHG3','IGHG4',
'S100A2',
'IFNG',
'CXCL14',
'IFNG',
'CD68',
'S100A9',
'FOXA1',
'ALDH1A1',
'C7',
'PI3K3CA', 'PTEN', 'MKI67', 'BRCA1','BRCA2']

cols = ['Segment','Gene','R2','PearsonCorr','PearsonQVals','VarianceRatio', 'ConcordanceCorr',
        'R2_calib','VarianceRatio_calib', 'ConcordanceCorr_calib']

In [ ]:
stats_of_interest = combined_scores_unique.loc[combined_scores_unique['Gene'].isin(genes_of_interest),cols]
stats_of_interest.to_csv(os.path.join(save_path,'genes_of_interest.csv'))

### PanCK+

In [ ]:
cond = (combined_scores_unique['R2']<0.0)
combined_scores_unique.loc[cond&(combined_scores_unique['Segment']=='CD45+'),cols].describe()

In [ ]:
cond = (combined_scores_unique['R2']<0.0)
combined_scores_unique.loc[cond&(combined_scores_unique['Segment']=='PanCK+'),cols].describe()

# Figure 2 - Main results

In [ ]:
fig_path = os.path.join(root_path,'figures')
os.makedirs(fig_path,exists_ok=True)


In [ ]:

y_pred_cd45 = y_pred_cd45.drop_duplicates().sort_index()
y_pred_panck = y_pred_panck.drop_duplicates().sort_index()
y_test_cd45 = y_test_cd45.drop_duplicates().sort_index()
y_test_panck = y_test_panck.drop_duplicates().sort_index()

y_pred_cd45 = y_pred_cd45[~y_pred_cd45.index.duplicated(keep="first")].sort_index()
y_pred_panck = y_pred_panck[~y_pred_panck.index.duplicated(keep="first")].sort_index()


In [ ]:
top_25_panck = combined_scores_unique[combined_scores_unique['Segment']=='PanCK+'].sort_values('R2', ascending=False)[:25]
top_25_cd45 = combined_scores_unique[combined_scores_unique['Segment']=='CD45+'].sort_values('R2', ascending=False)[:25]


In [ ]:
cols = ['Segment','Gene','R2','PearsonCorr']
top_25_cd45[cols].head()

In [ ]:
top_25_panck[cols].head()

In [ ]:
assert all(y_pred_cd45.index == y_test_cd45.index)
assert all(y_pred_panck.index == y_test_panck.index)

assert all(y_pred_cd45.columns == y_test_cd45.columns)
assert all(y_pred_panck.columns == y_test_panck.columns)

In [ ]:
y_test_cd45.head()

In [ ]:
def select_gene_at_percentile(df, percentile, stat='R2'):
    """
    Select gene whose R² is closest to the specified percentile.
    percentile: 0–100
    """
    r2_values = df[stat].values
    target_value = np.percentile(r2_values, percentile)
    
    # Find closest gene
    idx = np.abs(r2_values - target_value).argmin()
    gene = df.iloc[idx]["Gene"]
    actual_r2 = df.iloc[idx][stat]
    
    return gene, actual_r2

In [ ]:
df_cd45 = combined_scores_unique[combined_scores_unique["Segment"] == "CD45+"]
df_panck = combined_scores_unique[combined_scores_unique["Segment"] == "PanCK+"]
df = combined_scores_unique[['Segment','Gene','R2','PearsonCorr']].sort_values('R2',ascending=False)

In [ ]:
df.head()

In [ ]:
percentile_cd45 = 75
percentile_panck = 75

immune_top_gene, immune_r2 = select_gene_at_percentile(df_cd45, percentile_cd45)
tumour_mid_gene, epithelial_r2 = select_gene_at_percentile(df_panck, percentile_panck)

print(f"Immune (90th percentile): {immune_top_gene} | R² = {immune_r2:.3f}")
print(f"Epithelial (50th percentile): {Epithelial_mid_gene} | R² = {epithelial_r2:.3f}")

Representative genes selected at the 90th percentile (CD45+) and 50th percentile (PanCK+) of the gene-level R² distribution.

## V1

In [ ]:
import matplotlib as mpl
mpl.rcParams.update({
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "axes.linewidth": 0.8
})

sns.set_style("whitegrid")

In [ ]:
import matplotlib.gridspec as gridspec

palette = {"CD45+": "#355C7D", "PanCK+": "#C06C84"}


fig = plt.figure(figsize=(14, 5.5), constrained_layout=True)

gs = gridspec.GridSpec(
    2, 3,
    width_ratios=[1, 1, 0.9],
    height_ratios=[1, 1],
    figure=fig
)


axA  = fig.add_subplot(gs[:, 0])
axB  = fig.add_subplot(gs[:, 1])
axC1 = fig.add_subplot(gs[0, 2])
axC2 = fig.add_subplot(gs[1, 2])

# =========================
# Panel A — R²
# =========================

sns.violinplot(
    data=df,
    x="Segment",
    y="R2",
    hue="Segment",
    palette=palette,
    inner=None,
    cut=0.5,
    linewidth=0.8,
    legend=False,
    ax=axA
)

sns.boxplot(
    data=df,
    x="Segment",
    y="R2",
    hue="Segment",
    palette=palette,
    width=0.25,
    showcaps=True,
    showfliers=False,
    boxprops={'facecolor':'white', 'zorder':3},
    medianprops={'color':'black', 'linewidth':1},
    whiskerprops={'linewidth':1},
    legend=False,
    ax=axA
)

axA.text(0, -0.75, "n = 104", ha="center", fontsize=6)
axA.text(1, -0.75, "n = 115", ha="center", fontsize=6)

axA.set_ylabel("Gene-level $R^2$")
axA.set_xlabel("")
axA.set_title("A  Predictive performance", loc="left")

# cleaner limits
r2_min = df["R2"].min()
r2_max = df["R2"].max()

# margin = 0.05 * (r2_max - r2_min)

# axA.set_ylim(r2_min - margin, r2_max + margin)

upper = np.ceil(r2_max * 10) / 10
lower = np.floor(r2_min * 10) / 10
axA.set_ylim(lower - 0.05, upper + 0.05)

# median annotation
for i, seg in enumerate(["CD45+", "PanCK+"]):
    median = df[df["Segment"]==seg]["R2"].median()
    axA.text(i, median + 0.05, f"{median:.2f}",
             ha="center", fontsize=9, fontweight="bold")

sns.despine(ax=axA)

# =========================
# Panel B — PCC
# =========================

for seg in ["CD45+", "PanCK+"]:
    sns.kdeplot(
        data=df[df["Segment"]==seg],
        x="PearsonCorr",
        ax=axB,
        fill=True,
        alpha=0.25,
        linewidth=1.2,
        color=palette[seg],
        label=seg
    )

axB.set_xlim(-0.3, 1.0)
axB.set_xlabel("Pearson's ρ")
axB.set_ylabel("Density")
axB.set_title("B  Correlation distribution", loc="left")
axB.legend(frameon=False)

sns.despine(ax=axB)

# =========================
# Panel C — Scatter
# =========================

def scatter_panel(ax, gene, df_true, df_pred, label, percentile):

    y_true = df_true.loc[gene].values
    y_pred = df_pred.loc[gene].values
    
    r = np.corrcoef(y_true, y_pred)[0,1]
    r2 = r**2

    ax.scatter(y_true, y_pred,
               s=25,
               alpha=0.7,
               color="#355C7D",
               edgecolor="none")

    min_val = min(y_true.min(), y_pred.min())
    max_val = max(y_true.max(), y_pred.max())

    ax.plot([min_val, max_val],
            [min_val, max_val],
            linestyle="--",
            linewidth=1,
            color="black")

    ax.set_aspect("equal", adjustable="box")

    ax.set_xlabel("Measured")
    ax.set_ylabel("Predicted")
    ax.set_title(f"{label}  {gene} ({percentile}th %ile $R^2$)",
                 loc="left")

    ax.text(0.05, 0.92,
            f"ρ = {r:.2f}\n$R^2$ = {r2:.2f}",
            transform=ax.transAxes,
            verticalalignment='top',
            fontsize=9)

    sns.despine(ax=ax)

scatter_panel(axC1, immune_top_gene,
              y_test_cd45, y_pred_cd45,
              "C1  CD45+", percentile_cd45)

scatter_panel(axC2, tumour_mid_gene,
              y_test_panck, y_pred_panck,
              "C2  PanCK+", percentile_panck)

#plt.tight_layout()

fig.savefig(os.path.join(fig_path,"figure2-v1.pdf"), dpi=300, bbox_inches="tight")   # vector PDF for submission

plt.show()



## v2

In [ ]:
df = combined_scores_unique[['Segment','Gene','R2','PearsonCorr']].sort_values('R2',ascending=False)

# ── Publication settings ──────────────────────────────────────────────────────
mpl.rcParams.update({
    "font.family":        "Arial",          # Bioinformatics / ECCB standard
    "font.size":           7,
    "axes.titlesize":      8,
    "axes.labelsize":      7,
    "xtick.labelsize":     6,
    "ytick.labelsize":     6,
    "legend.fontsize":     6,
    "lines.linewidth":     0.8,
    "axes.linewidth":      0.6,
    "xtick.major.width":   0.6,
    "ytick.major.width":   0.6,
    "xtick.major.size":    2.5,
    "ytick.major.size":    2.5,
    "pdf.fonttype":        42,             # embeds fonts (required by most journals)
    "ps.fonttype":         42,
    "savefig.dpi":         300,
    "figure.dpi":          150,
    "axes.grid":            False
})

# Colorblind-friendly, print-safe palette (also works in greyscale)
palette = {"CD45+": "#2166AC", "PanCK+": "#B2182B"}

# ── Figure & GridSpec ─────────────────────────────────────────────────────────
# Double-column width = 7.0 in; height chosen to keep panels roughly square
fig = plt.figure(figsize=(7.0, 3.0), constrained_layout=True)


gs = gridspec.GridSpec(2, 3, width_ratios=[0.9, 1.2, 0.8], figure=fig)
gs_right = gridspec.GridSpecFromSubplotSpec(
    2, 1,
    subplot_spec=gs[:, 2],
    hspace=0.1   # only affects C1/C2, leaves A and B untouched
)
axA  = fig.add_subplot(gs[:, 0])
axB  = fig.add_subplot(gs[:, 1])
axC1 = fig.add_subplot(gs_right[0])
axC2 = fig.add_subplot(gs_right[1])


#axC1 = fig.add_subplot(gs[0, 2])
#axC2 = fig.add_subplot(gs[1, 2])

# ── Panel A — Gene-level R² ───────────────────────────────────────────────────
sns.violinplot(
    data=df,
    x="Segment", y="R2",
    hue="Segment",
    palette=palette,
    inner=None, 
    cut=0.3,
    linewidth=0.6,
    legend=False,
    ax=axA
)
sns.boxplot(
    data=df,
    x="Segment", y="R2",
    hue="Segment",
    palette=palette,
    width=0.3,
    showcaps=True, showfliers=False,
    boxprops={"facecolor": "white", "zorder": 3, "linewidth": 0.6},
    medianprops={"color": "black", "linewidth": 0.8},
    whiskerprops={"linewidth": 0.6},
    capprops={"linewidth": 0.6},
    legend=False,
    ax=axA
)

axA.axhline(y=0.2, color="grey", linewidth=0.6, linestyle="--", zorder=2)
#axA.text(1.48, 0.21, "$R^2$ = 0.2", 
#         ha="right", va="bottom", fontsize=6, color="black")
axA.text(-0.48, 0.1, "$R^2$ = 0.2",
         ha="left", va="bottom", fontsize=6, color="black")
         
axA.set_ylabel("Gene-level $R^2$")
axA.set_xlabel("")
axA.set_title("A  Predictive performance", loc="left", fontweight="bold")
axA.set_ylim(-0.8, 0.8)
axA.yaxis.set_major_locator(mpl.ticker.MultipleLocator(0.4))
counts = df.groupby("Segment").size()
axA.set_xticklabels([
    f"CD45+\n$n$ = {counts['CD45+']}",
    f"PanCK+\n$n$ = {counts['PanCK+']}"
])

# for i, seg in enumerate(["CD45+", "PanCK+"]):
#     med = df[df["Segment"] == seg]["R2"].median()
#     axA.text(i, med + 0.06, f"{med:.2f}",
#              ha="center", fontsize=6, fontweight="bold")

for i, seg in enumerate(["CD45+", "PanCK+"]):
    q1  = df[df["Segment"] == seg]["R2"].quantile(0.25)
    q3  = df[df["Segment"] == seg]["R2"].quantile(0.75)
    mid = (q1 + q3) / 2          # vertical center of the IQR box

    axA.text(i, mid, f"{df[df['Segment']==seg]['R2'].median():.2f}",
             ha="center", va="center",
             fontsize=6, fontweight="bold")

sns.despine(ax=axA)

# ── Panel B — Pearson correlation distribution ────────────────────────────────
for seg in ["CD45+", "PanCK+"]:
    sns.kdeplot(
        data=df[df["Segment"] == seg],
        x="PearsonCorr",
        ax=axB,
        fill=True, alpha=0.20,
        linewidth=0.9,
        color=palette[seg],
        label=seg
    )

axB.set_xlim(-0.3, 1.0)
axB.set_xlabel("Pearson's ρ")
axB.set_ylabel("Density")
axB.set_title("B  Correlation distribution", loc="left", fontweight="bold")
axB.legend(frameon=False, 
           #handlelength=1.2,    # width of the coloured bar (in font-size units)
           handleheight=1.2,    # height of the coloured bar
           handletextpad=0.6,   # gap between handle and text
           fontsize=6)
sns.despine(ax=axB)

# ── Panel C — Representative scatter plots ────────────────────────────────────
def scatter_panel(ax, gene, df_true, df_pred, label, percentile):
    y_true = df_true.loc[gene].values
    y_pred = df_pred.loc[gene].values

    r  = np.corrcoef(y_true, y_pred)[0, 1]
    r2 = r ** 2

    ax.scatter(y_true, y_pred,
               s=6, alpha=0.65,
               color="#2166AC",
               edgecolor="none",
               rasterized=True)        # rasterize points → smaller PDF

    lo = min(y_true.min(), y_pred.min())
    hi = max(y_true.max(), y_pred.max())
    ax.plot([lo, hi], [lo, hi],
            linestyle="--", linewidth=0.7, color="black")
    padding = (hi - lo) * 0.05
    ax.set_xlim(lo - padding, hi + padding)
    ax.set_ylim(lo - padding, hi + padding)

    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("Measured")
    ax.set_ylabel("Predicted")
    # ax.set_title(f"{label}  {gene} ({percentile}th %ile $R^2$)",
    #              loc="left", fontweight="bold")
    title = f"{label}\n{gene} ({percentile}th %ile $R^2$)"
    title = f"{label} {gene}"
    ax.set_title(title, loc="left", fontweight="bold")

    plt_txt = f"ρ = {r:.2f}\n$R^2$ = {r2:.2f}"

    ax.text(0.05, 0.98,
            plt_txt,
            transform=ax.transAxes,
            va="top", fontsize=6)

    sns.despine(ax=ax)

scatter_panel(axC1, immune_top_gene,
              y_test_cd45, y_pred_cd45,
              "C1  CD45+", percentile_cd45)

scatter_panel(axC2, tumour_mid_gene,
              y_test_panck, y_pred_panck,
              "C2  PanCK+", percentile_panck)

# ── Save ──────────────────────────────────────────────────────────────────────

#fig.savefig(os.path.join(fig_path,"figure2.eps"), dpi=300, bbox_inches="tight")   # vector eps
fig.savefig(os.path.join(fig_path,"figure2.pdf"), dpi=300, bbox_inches="tight")   # vector PDF for submission
plt.show()

## V3

In [ ]:
df = combined_scores_unique[['Segment','Gene','R2','PearsonCorr']].sort_values('R2',ascending=False)

# ── Publication settings ──────────────────────────────────────────────────────
mpl.rcParams.update({
    "font.family":        "Arial",
    "font.size":           7,
    "axes.titlesize":      8,
    "axes.labelsize":      7,
    "xtick.labelsize":     6,
    "ytick.labelsize":     6,
    "legend.fontsize":     6,
    "lines.linewidth":     0.8,
    "axes.linewidth":      0.6,
    "xtick.major.width":   0.6,
    "ytick.major.width":   0.6,
    "xtick.major.size":    2.5,
    "ytick.major.size":    2.5,
    "pdf.fonttype":        42,
    "ps.fonttype":         42,
    "savefig.dpi":         300,
    "figure.dpi":          150,
    "axes.grid":            False
})

palette = {"CD45+": "#2166AC", "PanCK+": "#B2182B"}

# ── Figure & GridSpec ─────────────────────────────────────────────────────────
fig = plt.figure(figsize=(4, 2.2), constrained_layout=True)
gs = gridspec.GridSpec(1, 2, width_ratios=[0.9, 1.2], figure=fig)
axA = fig.add_subplot(gs[0])
axB = fig.add_subplot(gs[1])

# ── Panel A — Gene-level R² ───────────────────────────────────────────────────
sns.violinplot(
    data=df,
    x="Segment", y="R2",
    hue="Segment",
    palette=palette,
    inner=None,
    cut=0.3,
    linewidth=0.6,
    legend=False,
    ax=axA
)
sns.boxplot(
    data=df,
    x="Segment", y="R2",
    hue="Segment",
    palette=palette,
    width=0.3,
    showcaps=True, showfliers=False,
    boxprops={"facecolor": "white", "zorder": 3, "linewidth": 0.6},
    medianprops={"color": "black", "linewidth": 0.8},
    whiskerprops={"linewidth": 0.6},
    capprops={"linewidth": 0.6},
    legend=False,
    ax=axA
)

axA.axhline(y=0.2, color="grey", linewidth=0.6, linestyle="--", zorder=2)
axA.text(-0.48, 0.1, "$R^2$ = 0.2",
         ha="left", va="bottom", fontsize=6, color="black")
axA.set_ylabel("Gene-level $R^2$")
axA.set_xlabel("")
axA.set_title("A  Predictive performance", loc="left", fontweight="bold")
axA.set_ylim(-0.8, 0.8)
axA.yaxis.set_major_locator(mpl.ticker.MultipleLocator(0.4))
counts = df.groupby("Segment").size()
axA.set_xticklabels([
    f"CD45+\n$n$ = {counts['CD45+']}",
    f"PanCK+\n$n$ = {counts['PanCK+']}"
])

for i, seg in enumerate(["CD45+", "PanCK+"]):
    q1  = df[df["Segment"] == seg]["R2"].quantile(0.25)
    q3  = df[df["Segment"] == seg]["R2"].quantile(0.75)
    mid = (q1 + q3) / 2
    axA.text(i, mid, f"{df[df['Segment']==seg]['R2'].median():.2f}",
             ha="center", va="center",
             fontsize=6, fontweight="bold")

sns.despine(ax=axA)

# ── Panel B — Pearson correlation distribution ────────────────────────────────
for seg in ["CD45+", "PanCK+"]:
    sns.kdeplot(
        data=df[df["Segment"] == seg],
        x="PearsonCorr",
        ax=axB,
        fill=True, alpha=0.20,
        linewidth=0.9,
        color=palette[seg],
        label=seg
    )

axB.set_xlim(-0.3, 1.0)
axB.set_xlabel("Pearson's ρ")
axB.set_ylabel("Density")
axB.set_title("B  Correlation distribution", loc="left", fontweight="bold")
axB.legend(frameon=False,
           handleheight=1.2,
           handletextpad=0.6,
           fontsize=6)
sns.despine(ax=axB)

# ── Save ──────────────────────────────────────────────────────────────────────
fig.savefig(os.path.join(fig_path, "figure2-single.pdf"), dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
df = combined_scores_unique[['Segment','Gene','R2','PearsonCorr']].sort_values('R2',ascending=False)
fig, ax = plt.subplots(1,1,figsize=(6,4))
axes={}
axes['A']=ax
sns.barplot(
    data=combined_scores_unique,
    x='Segment',
    y='PearsonCorr',
    hue='GeneSet',
    order=segment_order,
    hue_order=geneset_order,
    palette='viridis',
    errorbar='sd',
    ax=axes['A']
)
# Add value labels automatically
for container in axes['A'].containers:
    axes['A'].bar_label(container, fmt="%.2f", label_type='center',  color='white')

# Replace x-tick labels
axes['A'].set_xticks(range(len(segment_order)))  # Optional but useful

axes['A'].set_xticklabels(["Immune (CD45+)", "Tumour (PanCK+)"])

# Axis labels and title
axes['A'].set_ylabel("Pearson's ρ") #, fontsize=12)
axes['A'].set_xlabel("")
axes['A'].set_ylim(0.0, 1.0)
#axes[0].set_title("Correlation between predicted and measured expression in hold-out test set")

# Legend formatting
handles, labels = axes['A'].get_legend_handles_labels()
new_labels = ['Cancer Panel', 'Highly Variable Genes', 'Spatially Variable Genes']
axes['A'].legend(handles, new_labels, title="GeneSet", loc='upper right', frameon=True)



# Remaining Analyses

## Stats for Results

## Table1


Assess compartment difference

In [ ]:
metrics = ["R2", "PearsonCorr"]

In [ ]:
df = combined_scores_unique[['Segment','Gene','R2','PearsonCorr','PearsonQVals',	'PearsonCI_low',	'PearsonCI_high']].copy()
df['Robust'] = (df["R2"] > 0.2) & (df["PearsonQVals"] < 0.05)
df.head()

#### Mann-Whitney test

#### all scores

In [ ]:
def get_stats_dict(df):
    n = len(df)
    
    # --- R2 stats ---
    r2_median = df["R2"].median()
    r2_q1 = df["R2"].quantile(0.25)
    r2_q3 = df["R2"].quantile(0.75)
    r2_iqr = r2_q3 - r2_q1
    mean_r2 = df["R2"].mean()
    sd_r2 = df["R2"].std()
    
    n_r2_pos = (df["R2"] > 0).sum()
    pct_r2_pos = 100 * n_r2_pos / n
    n_r2_thres = (df["R2"] > 0.2).sum()
    pct_r2_thres = 100 * n_r2_thres / n
    
    # --- PCC stats ---
    pcc_median = df["PearsonCorr"].median()
    pcc_q1 = df["PearsonCorr"].quantile(0.25)
    pcc_q3 = df["PearsonCorr"].quantile(0.75)
    pcc_iqr = pcc_q3 - pcc_q1
    
    # --- Robust genes ---
    n_robust = df["Robust"].sum()
    pct_robust = 100 * n_robust / n
    
    n_qsig = (df["PearsonQVals"] < 0.05).sum()
    pct_qsig = 100 * n_qsig / n
    
    return {
        "n": n,
        "Median R2 (Q1–Q3)": f"{r2_median:.3f} ({r2_q1:.3f}–{r2_q3:.3f})",
        "Median PCC (Q1–Q3)": f"{pcc_median:.3f} ({pcc_q1:.3f}–{pcc_q3:.3f})",
        "PCC qval < 0.05": f"{n_qsig} ({pct_qsig:.1f}%)",
        "n (%) R2 > 0": f"{n_r2_pos} ({pct_r2_pos:.1f}%)",
        "n (%) Robust Genes": f"{n_robust} ({pct_robust:.1f}%)",
        "n (%) R2 > 0.2": f"{n_r2_thres} ({pct_r2_thres:.1f}%)",
        "Mean R2": mean_r2,
        "R2 SD": sd_r2,
        "Median R2": r2_median,
        "Median PCC": pcc_median,
    }

In [ ]:
summary = []
for segment, sub in df.groupby("Segment"):
    summary_dict = get_stats_dict(sub)
    summary_dict['Compartment']=segment
    summary.append(summary_dict)

summary_dict = get_stats_dict(df)
summary_dict['Compartment']=''
summary.append(summary_dict)


summary_table = pd.DataFrame(summary)
col = summary_table.pop('Compartment') 
summary_table.insert(0, 'Compartment', col) 
summary_table


In [ ]:
# Split groups
cd45 = df[df["Segment"] == "CD45+"]
panck = df[df["Segment"] == "PanCK+"]

# ----- R2 -----
u_r2, p_r2 = mannwhitneyu(
    cd45["R2"].dropna().to_numpy(),
    panck["R2"].dropna().to_numpy(),
    alternative="two-sided"
)

n1 = len(cd45)
n2 = len(panck)


# ----- PCC -----
u_pcc, p_pcc = mannwhitneyu(
    cd45["PearsonCorr"].dropna().to_numpy(),
    panck["PearsonCorr"].dropna().to_numpy(),
    alternative="two-sided"
)

cliff_r2 = cliffs_delta(cd45["R2"].dropna().to_numpy(), panck["R2"].dropna().to_numpy())
cliff_pcc = cliffs_delta(cd45["PearsonCorr"].dropna().to_numpy(), panck["PearsonCorr"].dropna().to_numpy())

print(f"R2: U ={u_r2} p = {p_r2:.2e} Effect size ={cliff_r2:.2f}")
print(f"R2: U ={u_pcc} p = {p_pcc:.2e} Effect size ={cliff_pcc:.2f}")

In [ ]:
stats_row = pd.DataFrame({
    "Compartment": ["Mann Whitney (CD45+ vs PanCK+)"],
    "n": [""],
    "Median R2 (Q1–Q3)": [f"p = {p_r2:.2e}, delta = {cliff_r2:.2f}"],
    "Median PCC (Q1–Q3)":[f"p = {p_pcc:.2e}, delta = {cliff_pcc:.2f}"],
    "PCC qval < 0.05": [""],
    "n (%) R2 > 0":  [""],
    "n (%) Robust Genes": [""],
    "n (%) R2 > 0.2": [""],
    "Mean R2": [""],
    "R2 SD": [""],
    "Median R2": [""],
    "Median PCC": [""],
})

final_table = pd.concat([summary_table, stats_row], ignore_index=True)

final_table

In [ ]:
tab_path = os.path.join(root_path,'tables')
os.makedirs(tab_path,exists_ok=True)
final_table.to_csv(os.path.join(tab_path,'results_summary.csv'))

In [ ]:

for m in metrics:
    cd = df.loc[df["Segment"] == "CD45+", m].dropna().to_numpy()
    pk = df.loc[df["Segment"] == "PanCK+", m].dropna().to_numpy()

    u = mannwhitneyu(cd, pk, alternative="two-sided")
    d = (2 * u.statistic) / (len(cd) * len(pk)) - 1
    magnitude = "negligible" if abs(d) < 0.147 else "small" if abs(d) < 0.33 else "medium" if abs(d) < 0.474 else "large"

    print(f"\n{m}")
    print(f"  n_CD45={len(cd)}  n_PanCK={len(pk)}")
    print(f"  CD45   median={np.median(cd):.4f}  mean={np.mean(cd):.4f}")
    print(f"  PanCK  median={np.median(pk):.4f}  mean={np.mean(pk):.4f}")
    print(f"  U p-value={u.pvalue:.4g}  Cliff's δ={d:.3f} ({magnitude})")

In [ ]:
for m in metrics:
    cd = combined_scores_unique.loc[combined_scores_unique["Segment"] == "CD45+", m].dropna().to_numpy()
    pk = combined_scores_unique.loc[combined_scores_unique["Segment"] == "PanCK+", m].dropna().to_numpy()

    u = mannwhitneyu(cd, pk, alternative="two-sided")
    d = (2 * u.statistic) / (len(cd) * len(pk)) - 1
    magnitude = "negligible" if abs(d) < 0.147 else "small" if abs(d) < 0.33 else "medium" if abs(d) < 0.474 else "large"

    print(f"\n{m}")
    print(f"  n_CD45={len(cd)}  n_PanCK={len(pk)}")
    print(f"  CD45   median={np.median(cd):.4f}  mean={np.mean(cd):.4f}")
    print(f"  PanCK  median={np.median(pk):.4f}  mean={np.mean(pk):.4f}")
    print(f"  U p-value={u.pvalue:.4g}  Cliff's δ={d:.3f} ({magnitude})")

### Good PCC but poor R2

only 2 with good PCC but negative R2

In [ ]:
cond = (combined_scores_unique['R2']<0.0)
cond = cond & (combined_scores_unique['PearsonCorr']>0.5)
combined_scores_unique.loc[cond&(combined_scores_unique['Segment']=='PanCK+'),cols].describe()

In [ ]:
combined_scores_unique.loc[cond&(combined_scores_unique['Segment']=='PanCK+'),cols]

In [ ]:
cond = (combined_scores_unique['R2']<0.2)
cond = cond & (combined_scores_unique['R2']>0)
combined_scores_unique.loc[cond&(combined_scores_unique['Segment']=='PanCK+'),cols].describe()

In [ ]:

cond =  (combined_scores_unique['GeneSet']=='Panel')
cond = cond&(combined_scores_unique['Segment']=='PanCK+')
print(f"total PanCK panel: {combined_scores_unique[cond].sort_values('R2').shape[0]}")

Negative R2

In [ ]:
cond = (combined_scores_unique['R2']<0)
combined_scores_unique.loc[cond,['GeneSet']+cols].sort_values('R2')

### Good predictors

#### Distribution of PCC

In [ ]:
geneset_good_models.loc[geneset_good_models['Segment']=='CD45+','PearsonCorr'].describe()

In [ ]:
geneset_good_models.loc[geneset_good_models['Segment']=='PanCK+','PearsonCorr'].describe()

# dONT HARD CODE GENE COUNTS

In [ ]:
cond = ((geneset_good_models['PearsonCorr']>0.5) &
        (geneset_good_models['Segment']=='CD45+')
        )
max_cd45_count = geneset_good_models.loc[cond,'PearsonCorr'].count()
print(f"{max_cd45_count/104*100:.2f}% ({max_cd45_count} of 104) of the CD45+ gene \
panel surpassed the R2 threshold (0.2) with statistically \
significant Pearson’s correlation above 0.5")

In [ ]:
cond = ((geneset_good_models['PearsonCorr']>0.5) &
        (geneset_good_models['Segment']=='PanCK+')
        )
max_panck_count = geneset_good_models.loc[cond,'PearsonCorr'].count()

print(f"while only {max_panck_count/115*100:.2f}% ({max_panck_count} of 115) of PanCK+ gene panel achieved this level of significance.")

In [ ]:
geneset_good_models.to_csv(os.path.join(summary_path,'genesetgoodmodels.csv'))

In [ ]:
geneset_good_models=geneset_good_models.sort_values('PearsonCorr', ascending=False)


save the y_test and y_pred for only good predictors

In [ ]:
unique_genes={}
for segment in ['CD45+','PanCK+']:
    print(segment, feat_extr)
    good_genes = list(geneset_good_models.loc[geneset_good_models['Segment']==segment,'Gene'].values)
    y_pred_seg = y_pred_comb_dict[(feat_extr, segment, 'SVG')]
    #drop duplicates
    y_pred_seg = y_pred_seg.join(y_pred_comb_dict[(feat_extr, segment, 'HVG')], how='left',rsuffix='_drop')
    y_pred_seg = y_pred_seg.join(y_pred_comb_dict[(feat_extr, segment, 'Panel')], how='left',rsuffix='_drop')
    y_pred_seg = y_pred_seg.drop(columns=y_pred_seg.filter(like='_drop').columns)

    y_pred_seg.to_csv(os.path.join(pred_path,f'preds_{segment[:-1]}_{feat_extr}_SVM_goodpredictors.csv'))

## Correlation Plots

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

### Combined Correlation Figures

In [ ]:
SUPTITLESIZE=18
TITLESIZE=16
LABELSIZE=14
NOTESIZE=10

In [ ]:


plt.figure(figsize=(7, 5))
ax = sns.barplot(
    data=combined_scores_unique,#seg_test_gene_scores,
    x='Segment',
    y='PearsonCorr',
    hue='GeneSet',
    order=segment_order,
    hue_order=geneset_order,
    palette='viridis',
    errorbar='sd'
)
# Add value labels automatically
for container in ax.containers:
    ax.bar_label(container, fmt="%.2f", label_type='center', fontsize=14, color='white')

# Replace x-tick labels
ax.set_xticklabels(["Immune Compartments (CD45⁺)", "Epithelial Compartments (PanCK⁺)"], fontsize=12)

# Axis labels and title
ax.set_ylabel("PearsonCorrelation Coefficient (ρ)", fontsize=12)
ax.set_xlabel("")
ax.set_ylim(0.0, 1.0)
plt.title("Correlation between predicted and measured expression in hold-out test set")

# Legend formatting
handles, labels = ax.get_legend_handles_labels()
new_labels = ['Cancer Panel', 'Highly Variable Genes', 'Spatially Variable Genes']
ax.legend(handles, new_labels, title="GeneSet", loc='upper right', frameon=True)


# for i, (val, qval) in enumerate(zip(
#     seg_test_gene_scores["PearsonCorr"],
#     seg_test_gene_scores["PearsonQVals"]
# )):
#     if qval <= 0.001:
#         sig = "***"
#     elif qval <= 0.01:
#         sig = "**"
#     elif qval <= 0.05:
#         sig = "*"
#     else:
#         sig = ""
#     xpos = val + 0.01  
#     ax.text(xpos, i, sig, va='center', fontsize=10, fontweight='bold')
#
plt.tight_layout(rect=[0, 0.01, 1, 0.9])
plt.title("Correlation of predicted and measured expression", fontsize=18)

plt.figtext(0.5, 0.0, '* p < 0.05 | ** p < 0.01 | *** p < 0.001', ha='center', fontsize=10)
plt.savefig("pcc_correlation_plot.pdf", format='pdf', bbox_inches='tight')  
plt.show()

In [ ]:
use_calibrated = True
fig, axes = plt.subplots( 1, 2, figsize=(12,5))

df = combined_scores_unique  #seg_test_gene_scores,
if use_calibrated:
    suffix='_calib'
else:
    suffix=''
### A

sns.barplot(
    data=df,
    x='Segment',
    y=f'PearsonCorr{suffix}',
    hue='GeneSet',
    order=segment_order,
    hue_order=geneset_order,
    palette='viridis',
    errorbar='sd',
    ax=axes[0]
)
# Add value labels automatically
for container in axes[0].containers:
    axes[0].bar_label(container, fmt="%.2f", label_type='center',  color='white')

# Replace x-tick labels
axes[0].set_xticks(range(len(segment_order)))  # Optional but useful

axes[0].set_xticklabels(["Immune (CD45⁺)", "Epithelial (PanCK⁺)"], fontsize=LABELSIZE)

# Axis labels and title
axes[0].set_ylabel("Pearson's ρ", fontsize=LABELSIZE) #, fontsize=12)
axes[0].set_xlabel("")
axes[0].set_ylim(0.0, 1.0)
#axes[0].set_title("Correlation between predicted and measured expression in hold-out test set")

# Legend formatting
handles, labels = axes[0].get_legend_handles_labels()
new_labels = ['Cancer Panel', 'Highly Variable Genes', 'Spatially Variable Genes']
axes[0].legend(handles, new_labels, title="GeneSet", loc='upper right', frameon=True)


### B
sns.histplot(
    data=df,
    x=f'PearsonCorr{suffix}',
    hue='Segment',
    bins=20,
    element='bars',  # or "bars"
    common_norm=False,  # don’t normalize across groups
    palette='viridis',
    edgecolor='black',
    ax=axes[1]
)
#axes[1].set_title("Distribution of correlation between measured and predicted expression")
axes[1].set_xlabel("Pearson's ρ", fontsize=LABELSIZE)
axes[1].set_ylabel("Count", fontsize=LABELSIZE)

## cant use get_legend_handles... for some reason
legend = axes[0].get_legend()
if legend:
    labels = ['Immune (CD45⁺)', 'Epithelial (PanCK⁺)']
    for text, new_label in zip(legend.get_texts(), labels):
        text.set_text(new_label)

fig.suptitle('Correlation between measured and predicted expression', ha='center', fontsize=SUPTITLESIZE)
plt.figtext(0.5, 0.0, '* p < 0.05 | ** p < 0.01 | *** p < 0.001', ha='center', fontsize=NOTESIZE)

### Prepare for paper
plt.tight_layout()
plt.subplots_adjust(wspace=0.3)
axes[0].text(-0.1, 1.05, 'A', transform=axes[0].transAxes,
             fontsize=14,  va='top', ha='center')

axes[1].text(-0.1, 1.05, 'B', transform=axes[1].transAxes,
             fontsize=14, va='top', ha='center')

#plt.savefig("pcc_correlation_plot.pdf", format='pdf', bbox_inches='tight')  



plt.show()



### Correlation Coefficient Chart

In [ ]:

fig, axes = plt.subplots( 1, 2, figsize=(12,5))

df = combined_scores_unique  #seg_test_gene_scores,
### A

sns.barplot(
    data=df,
    x='Segment',
    y='ConcordanceCorr_calib',
    hue='GeneSet',
    order=segment_order,
    hue_order=geneset_order,
    palette='viridis',
    errorbar='sd',
    ax=axes[0]
)
# Add value labels automatically
for container in axes[0].containers:
    axes[0].bar_label(container, fmt="%.2f", label_type='center',  color='white')

# Replace x-tick labels
axes[0].set_xticks(range(len(segment_order)))  # Optional but useful

axes[0].set_xticklabels(["Immune (CD45⁺)", "Tumour (PanCK⁺)"], fontsize=LABELSIZE)

# Axis labels and title
axes[0].set_ylabel("CCC", fontsize=LABELSIZE) #, fontsize=12)
axes[0].set_xlabel("")
axes[0].set_ylim(0.0, 1.0)
#axes[0].set_title("Correlation between predicted and measured expression in hold-out test set")

# Legend formatting
handles, labels = axes[0].get_legend_handles_labels()
new_labels = ['Cancer Panel', 'Highly Variable Genes', 'Spatially Variable Genes']
axes[0].legend(handles, new_labels, title="GeneSet", loc='upper right', frameon=True)


### B
sns.histplot(
    data=df,
    x='ConcordanceCorr_calib',
    hue='Segment',
    bins=20,
    element='bars',  # or "bars"
    common_norm=False,  # don’t normalize across groups
    palette='viridis',
    edgecolor='black',
    ax=axes[1]
)

axes[1].set_xlabel("CCC", fontsize=LABELSIZE)
axes[1].set_ylabel("Count", fontsize=LABELSIZE)

## cant use get_legend_handles... for some reason
legend = axes[0].get_legend()
if legend:
    labels = ['Immune (CD45⁺)', 'Tumour (PanCK⁺)']
    for text, new_label in zip(legend.get_texts(), labels):
        text.set_text(new_label)

fig.suptitle(f'Concordance Correlation - {CALIBRATION_FRACTION} Calibration Set', ha='center', fontsize=SUPTITLESIZE)

### Prepare for paper
plt.tight_layout()
plt.subplots_adjust(wspace=0.3)
axes[0].text(-0.1, 1.05, 'A', transform=axes[0].transAxes,
             fontsize=14,  va='top', ha='center')

axes[1].text(-0.1, 1.05, 'B', transform=axes[1].transAxes,
             fontsize=14, va='top', ha='center')




plt.show()



## Other Analysis

In [ ]:
geneset_good_models[geneset_good_models['Segment']=='CD45+'].describe()

In [ ]:
geneset_good_models[geneset_good_models['Segment']=='PanCK+'].describe()

In [ ]:

good_genes_panck = sorted(list(geneset_good_models.loc[geneset_good_models['Segment']=='PanCK+','Gene'].unique()))
good_genes_cd45 = sorted(list(geneset_good_models.loc[geneset_good_models['Segment']=='CD45+','Gene'].unique()))

all models with R2 > 0.2 have a statistically significant Pearson correlation (FDR-adjusted)

In [ ]:
geneset_good_models.describe()

## Scatterplots

In [ ]:
top_25_panck = combined_scores_unique[combined_scores_unique['Segment']=='PanCK+'].sort_values('PearsonCorr', ascending=False)[:25]
top_25_cd45 = combined_scores_unique[combined_scores_unique['Segment']=='CD45+'].sort_values('PearsonCorr', ascending=False)[:25]


Examine the most highly correlated genes

In [ ]:
best_gene_cd45 = top_25_cd45['Gene'].iloc[0]
best_gene_panck = top_25_panck['Gene'].iloc[0]
print(f"best genes: {best_gene_cd45} | {best_gene_panck}")

Some genes appear in both HVG and SVG genesets - they achieve identical predictions in each as each gene in a geneset is modelled independently (using the same seed). We want to examine per-gene so we need to drop the duplicates within each segment (some genes still appear in both segments as they can be expressed by both immune and tumour cells)

In [ ]:
y_train_cd45 = y_train_cd45.drop_duplicates().sort_index()
y_trainpred_cd45 = y_trainpred_cd45.drop_duplicates().sort_index()
y_train_panck = y_train_panck.drop_duplicates().sort_index()
y_trainpred_panck = y_trainpred_panck.drop_duplicates().sort_index()
#test
y_test_cd45 = y_test_cd45.drop_duplicates().sort_index()
y_pred_cd45 = y_pred_cd45.drop_duplicates().sort_index()
y_test_panck = y_test_panck.drop_duplicates().sort_index()
y_pred_panck = y_pred_panck.drop_duplicates().sort_index()
#calibration
y_pred_calib_cd45 = y_pred_calib_cd45.drop_duplicates().sort_index()
y_pred_calib_panck = y_pred_calib_panck.drop_duplicates().sort_index()
y_pred_calib_cd45 = y_pred_calib_cd45.loc[~y_pred_calib_cd45.index.duplicated(keep='first'), :]
y_pred_calib_panck = y_pred_calib_panck.loc[~y_pred_calib_panck.index.duplicated(keep='first'), :]

In [ ]:
y_pred_cd45.to_csv(os.path.join(save_path,'y_pred_cd45.csv'))
y_pred_panck.to_csv(os.path.join(save_path,'y_pred_panck.csv'))
y_pred_calib_cd45.to_csv(os.path.join(save_path,'y_pred_calib_cd45.csv'))
y_pred_calib_panck.to_csv(os.path.join(save_path,'y_pred_calib_panck.csv'))

In [ ]:
y_pred_panck.shape

In [ ]:
y_test_cd45.to_csv(os.path.join(save_path,'y_test_cd45.csv'))
y_test_panck.to_csv(os.path.join(save_path,'y_test_panck.csv'))

### CD45+

In [ ]:
pred_scatterplot(y_train_cd45,y_trainpred_cd45,geneset_good_models,
                 genelist=list(top_25_cd45['Gene'][:10]), 
                 title="TRAIN: Immune Segments (CD45+)" )

In [ ]:
pred_scatterplot(y_test_cd45,y_pred_cd45,geneset_good_models,
                 genelist=list(top_25_cd45['Gene'][:10]), 
                 title="Immune Segments (CD45+)" )


In [ ]:
pred_scatterplot(y_test_cd45,y_pred_calib_cd45,geneset_good_models,
                 genelist=list(top_25_cd45['Gene'][:10]), 
                 title="CALIBRATED: Immune Segments (CD45+)" )

### Supp Figure - Scatterplots

In [ ]:
import matplotlib.gridspec as gridspec

def plot_gene(ax, y_measured, y_pred, scores, gene):
    qval = scores.loc[scores['Gene'] == gene, 'PearsonQVals'].values[0]

    sns.scatterplot(
        x=y_measured.loc[gene, :],
        y=y_pred.loc[gene, :],
        ax=ax,
        s=20,
        alpha=0.7
    )

    # identity line
    lims = [
        min(ax.get_xlim()[0], ax.get_ylim()[0]),
        max(ax.get_xlim()[1], ax.get_ylim()[1])
    ]
    ax.plot(lims, lims, '--', color='black', linewidth=1)

    ax.text(
        0.95, 0.05,
        f"q={qval:.2e}",
        transform=ax.transAxes,
        fontsize=7,
        ha='right',
        va='bottom',
        bbox=dict(facecolor='grey', edgecolor='none', alpha=0.7)
    )

    ax.set_xlabel("Measured Expression",fontsize=9)
    ax.set_ylabel("Predicted Expression",fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(r"$\it{" + gene + "}$", fontsize=9)


def make_combined_figure(
    y_measured,
    y_pred_raw,
    y_pred_calib,
    scores,
    genelist,
    savepath="scatter_comparison.pdf"
):

    n_genes = len(genelist)
    ncols = 5
    nrows = int(np.ceil(n_genes / ncols))

    fig = plt.figure(figsize=(12, 6), constrained_layout=True)

    outer = gridspec.GridSpec(2, 1, figure=fig, height_ratios=[1, 1], hspace=0.25)

    # ---- RAW GRID ----
    gs_raw = gridspec.GridSpecFromSubplotSpec(
        nrows, ncols,
        subplot_spec=outer[0],
        wspace=0.3, hspace=0.4
    )

    for i, gene in enumerate(genelist):
        row = i // ncols
        col = i % ncols
        ax = fig.add_subplot(gs_raw[row, col])
        plot_gene(ax, y_measured, y_pred_raw, scores, gene)

    fig.text(0.5, 0.95, "Raw Expression",
             ha='center', fontsize=12)

    # ---- CALIBRATED GRID ----
    gs_cal = gridspec.GridSpecFromSubplotSpec(
        nrows, ncols,
        subplot_spec=outer[1],
        wspace=0.3, hspace=0.4
    )

    for i, gene in enumerate(genelist):
        row = i // ncols
        col = i % ncols
        ax = fig.add_subplot(gs_cal[row, col])
        plot_gene(ax, y_measured, y_pred_calib, scores, gene)

    fig.text(0.5, 0.48, "Calibrated Expression",
             ha='center', fontsize=12)

    #plt.tight_layout(rect=[0, 0, 1, 0.97])

    fig.savefig(savepath, format="pdf")
    #plt.close(fig)
    plt.show()

In [ ]:
def pred_scatterplot_save(y_measured, 
                    y_pred, 
                    scores, 
                    genelist, 
                    title="Predicted vs Measured", 
                    num_genes=10,
                    save_path='scatter.pdf',
                    panel_label='A',
                    fig=None,        
                    axs=None      
                    ):

    ncols = 5
    nrows = int(num_genes / ncols)

    # Use provided fig/axs or create new ones
    if fig is None or axs is None:
        fig, axs = plt.subplots(nrows, ncols, figsize=(10, nrows*2), sharex=False, sharey=False)
    
    axs = axs.flatten()

    for i, genename in enumerate(genelist):
        qval = scores.loc[scores['Gene']==genename,'PearsonQVals'].values[0]
        pval = scores.loc[scores['Gene']==genename,'PearsonPVals'].values[0]
        sns.scatterplot(x=y_measured.loc[genename,:], y=y_pred.loc[genename,:], ax=axs[i])

        lims = [
            min(axs[i].get_xlim()[0], axs[i].get_ylim()[0]),
            max(axs[i].get_xlim()[1], axs[i].get_ylim()[1])
        ]
        axs[i].plot(lims, lims, '--', color='black', linewidth=1)
        axs[i].text(
            0.95, 0.05, f"p={qval:.2e}",
            transform=axs[i].transAxes,
             ha='right', va='bottom', #fontsize=8,
            bbox=dict(facecolor='grey', edgecolor='none', alpha=0.6)
        )
        axs[i].set_aspect('equal', adjustable='box')
        axs[i].set_xticks([])
        axs[i].set_yticks([])
        axs[i].set_xlabel("Measured Expression") #, fontsize=9
        axs[i].set_ylabel("Predicted Expression") #, fontsize=9
        axs[i].set_title(r"$\it{" + genename + "}$")

    # Row label instead of suptitle when embedded in combined figure
    axs[0].annotate(panel_label, xy=(0,1), xycoords='axes fraction',
                    xytext=(-axs[0].yaxis.labelpad - 30, 0),
                    textcoords='offset points',fontsize=9, 
                    fontweight='bold', ha='right', va='center')

    axs[0].annotate(title, xy=(0,0), xycoords='axes fraction',
                    xytext=(-axs[0].yaxis.labelpad - 30, 0),
                    textcoords='offset points',fontsize=9, 
                    fontweight='bold', ha='right', va='center', rotation=90)

    if save_path:
        plt.tight_layout()
        fig.savefig(save_path, format="pdf")
        plt.show()

    return fig, axs

In [ ]:
mpl.rcParams.update({
    "font.family":        "Arial",          # Bioinformatics / ECCB standard
    "font.size":           7,
    "axes.titlesize":      8,
    "axes.labelsize":      7,
    "xtick.labelsize":     6,
    "ytick.labelsize":     6,
    "legend.fontsize":     6,
    "lines.linewidth":     0.8,
    "axes.linewidth":      0.6,
    "xtick.major.width":   0.6,
    "ytick.major.width":   0.6,
    "xtick.major.size":    2.5,
    "ytick.major.size":    2.5,
    "pdf.fonttype":        42,             # embeds fonts (required by most journals)
    "ps.fonttype":         42,
    "savefig.dpi":         300,
    "figure.dpi":          150,
    "axes.grid":            False
})

In [ ]:
def scatter_segment(y_test, y_pred_raw, y_pred_calib, genes,segment='CD45',title=None):
    # Force square subplots by making figure height match the aspect
    ncols = 5
    nrows_per_group = 2
    fig_width = 8 #ncols * subplot_size + 1          # +1 for margins
    subplot_size = fig_width / ncols 
    gap =0.25          # gap between groups in inches
    n_groups = 2

    fig_height = (n_groups * nrows_per_group * subplot_size) + gap + 0.6

    fig = plt.figure(figsize=(fig_width, fig_height))

    gs = fig.add_gridspec(
        nrows_per_group * 2 + 1, ncols,
        hspace=0.1, wspace=0.3,
        height_ratios=[1, 1, gap, 1, 1],  # gap row scaled to subplot size
        left=0.05, right=0.98, top=0.95, bottom=0.04
    )

    # Build axes — skip row index 2 (the gap row)
    axs_raw  = np.array([[fig.add_subplot(gs[r, c]) for c in range(ncols)] for r in [0, 1]])
    axs_calib = np.array([[fig.add_subplot(gs[r, c]) for c in range(ncols)] for r in [3, 4]])

    # Draw the divider line
    line = plt.Line2D([0, 1], [0.5, 0.5], transform=fig.transFigure,
                    color='grey', linewidth=1)
    fig.add_artist(line)

    # Plot both groups (save_path=None to skip individual saves)
    pred_scatterplot_save(y_test, y_pred_raw, geneset_good_models,
                        genelist=genes,
                        title="Raw Predicted Expression",
                        save_path=None,panel_label='A',
                        fig=fig, axs=axs_raw)

    pred_scatterplot_save(y_test, y_pred_calib, geneset_good_models,
                        genelist=genes,
                        title="Calibrated Predicted Expression",
                        save_path=None,panel_label='B',
                        fig=fig, axs=axs_calib)
    if title: plt.suptitle(title, fontsize=11, y=1.01)
    plt.savefig(os.path.join(fig_path, f"suppfig-scatter_{segment}.pdf"), format="pdf", bbox_inches='tight')
    plt.show()

In [ ]:
scatter_segment(y_test_cd45, y_pred_cd45, y_pred_calib_cd45, list(top_25_cd45['Gene'][:10]),segment='cd45',
                title="Comparison of Predicted and Measured Expression in Immune (CD45+) compartments")

In [ ]:
scatter_segment(y_test_panck, y_pred_panck, y_pred_calib_panck, list(top_25_panck['Gene'][:10]),segment='panck',
                title="Comparison of Predicted and Measured Expression in Tumour (PanCK+) compartments")

### PanCK+

In [ ]:
pred_scatterplot(y_train_panck,y_trainpred_panck,geneset_good_models,
                 genelist=list(top_25_panck['Gene'][:10]), 
                 title="TRAIN: Tumour Segments (PanCK+)" )


In [ ]:
pred_scatterplot(y_test_panck,y_pred_panck,geneset_good_models,
                 genelist=list(top_25_panck['Gene'][:10]), 
                 title="Tumour Segments (PanCK+)" )

In [ ]:
pred_scatterplot(y_test_panck,y_pred_calib_panck,geneset_good_models,
                 genelist=list(top_25_panck['Gene'][:10]), 
                 title="CALIBRATED: Tumour Segments (PanCK+)" )

## Assess Variance  

Scatterplots show good trends but lower variation in predictetd expression than in measured expression.

confirm this statistically for 'good' predictors- per gene pred variance vs per gene test variance

### Train Variance

In [ ]:
gene_variance_panck = pd.DataFrame(data={'measured_std':y_train_panck.loc[good_genes_panck,:].T.std(),
                                         'predicted_std':y_trainpred_panck .loc[good_genes_panck,:].T.std()})
gene_variance_panck.insert(0,'Segment','PanCK+')

In [ ]:
y_trainpred_cd45 = y_trainpred_cd45[~y_trainpred_cd45.index.duplicated(keep='first')]

In [ ]:
gene_variance_cd45 = pd.DataFrame(data={'measured_std':y_train_cd45.loc[good_genes_cd45,:].T.std(),
                                         'predicted_std':y_trainpred_cd45.loc[good_genes_cd45,:].T.std()})

In [ ]:


gene_variance_cd45.insert(0,'Segment','CD45+')
gene_variance_train = pd.concat([gene_variance_cd45,gene_variance_panck])

In [ ]:
gene_variance_train['difference'] = gene_variance_train['measured_std']-gene_variance_train['predicted_std']
gene_variance_train['ratio'] = gene_variance_train['predicted_std'] / gene_variance_train['measured_std']
gene_variance_train.describe()

### Test Variance

In [ ]:
y_pred_cd45 = y_pred_cd45[~y_pred_cd45.index.duplicated(keep='first')]

In [ ]:
gene_variance_panck = pd.DataFrame(data={'measured_std':y_test_panck.loc[good_genes_panck,:].T.std(),
                                         'predicted_std':y_pred_panck.loc[good_genes_panck,:].T.std()})
gene_variance_panck.insert(0,'Segment','PanCK+')
gene_variance_cd45 = pd.DataFrame(data={'measured_std':y_test_cd45.loc[good_genes_cd45,:].T.std(),
                                         'predicted_std':y_pred_cd45.loc[good_genes_cd45,:].T.std()})
gene_variance_cd45.insert(0,'Segment','CD45+')
gene_variance = pd.concat([gene_variance_cd45,gene_variance_panck])
gene_variance.head()

In [ ]:
gene_variance_calib_panck = pd.DataFrame(data={'measured_std':y_test_panck.loc[good_genes_panck,:].T.std(),
                                         'predicted_std':y_pred_calib_panck.loc[good_genes_panck,:].T.std()})
gene_variance_calib_panck.insert(0,'Segment','PanCK+')
gene_variance_calib_cd45 = pd.DataFrame(data={'measured_std':y_test_cd45.loc[good_genes_cd45,:].T.std(),
                                         'predicted_std':y_pred_calib_cd45.loc[good_genes_cd45,:].T.std()})
gene_variance_calib_cd45.insert(0,'Segment','CD45+')
gene_variance_calib = pd.concat([gene_variance_calib_cd45,gene_variance_calib_panck])
gene_variance_calib.head()

In [ ]:
gene_variance['difference'] = gene_variance['measured_std']-gene_variance['predicted_std']
gene_variance['ratio'] = gene_variance['predicted_std'] / gene_variance['measured_std']
gene_variance.describe()

In [ ]:
gene_variance_calib['difference'] = gene_variance_calib['measured_std']-gene_variance_calib['predicted_std']
gene_variance_calib['ratio'] = gene_variance_calib['predicted_std'] / gene_variance_calib['measured_std']
gene_variance_calib['ratio'].describe()

In [ ]:
gene_variance_panck_all = pd.DataFrame(data={'measured_std':y_test_panck.T.std(),
                                         'predicted_std':y_pred_panck.T.std()})
gene_variance_panck_all.insert(0,'Segment','PanCK+')
gene_variance_cd45_all = pd.DataFrame(data={'measured_std':y_test_cd45.T.std(),
                                         'predicted_std':y_pred_cd45.T.std()})
gene_variance_cd45_all.insert(0,'Segment','CD45+')
gene_variance_all = pd.concat([gene_variance_cd45_all,gene_variance_panck_all])
gene_variance_all.head()

In [ ]:
gene_variance_calib_panck_all = pd.DataFrame(data={'measured_std':y_test_panck.T.std(),
                                         'predicted_std':y_pred_calib_panck.T.std()})
gene_variance_calib_panck_all.insert(0,'Segment','PanCK+')
gene_variance_calib_cd45_all = pd.DataFrame(data={'measured_std':y_test_cd45.T.std(),
                                         'predicted_std':y_pred_calib_cd45.T.std()})
gene_variance_calib_cd45_all.insert(0,'Segment','CD45+')
gene_variance_calib_all = pd.concat([gene_variance_calib_cd45_all,gene_variance_calib_panck_all])
gene_variance_calib_all.head()

In [ ]:

gene_variance_panck_train_all = pd.DataFrame(data={'measured_std':y_train_panck.T.std(),
                                         'predicted_std':y_trainpred_panck.T.std()})
gene_variance_panck_train_all.insert(0,'Segment','PanCK+')
gene_variance_cd45_train_all = pd.DataFrame(data={'measured_std':y_train_cd45.T.std(),
                                         'predicted_std':y_trainpred_cd45.T.std()})
gene_variance_cd45_train_all.insert(0,'Segment','CD45+')
gene_variance_train_all = pd.concat([gene_variance_cd45_train_all,gene_variance_panck_train_all])
gene_variance_train_all.head()

### Evaluate Variance Ratio per segment

for each gene g I would define a ratio 
- \zeta_g = var(\hat{y}_cal)/var(y) say on the test set, with 
- \hat{y}_cal calibrated predicted expression for that gene on the test set 
- y ground-truth expressions for that gene on the test set

If variance shrinkage is still there you have \zeta_g<1, and we would like it to be close to 1 instead.

Assess overlap

In [ ]:
cd = set(
    gene_variance_train.loc[
        gene_variance_train["Segment"] == "CD45+"
    ].index
)

pk = set(
    gene_variance_train.loc[
        gene_variance_train["Segment"] == "PanCK+"
    ].index
)

overlap = cd & pk

print("CD45+ genes:", len(cd))
print("PanCK+ genes:", len(pk))
print("Overlap:", len(overlap))
print(f"Jaccard overlap: {len(overlap) / len(cd | pk):.2f}")

minimal overlap between segment genes so treat CD45+ and PanCK+ as two separate gene populations

compare unpaired

In [ ]:
gene_variance_train.groupby('Segment')[[
    'measured_std', 
    'predicted_std', 
    'difference', 
    'ratio'
]].mean()


Train

In [ ]:
gene_variance_train.groupby('Segment')[['ratio']].describe()

Test

In [ ]:
gene_variance.groupby('Segment')[['ratio']].describe()

Calibrated

In [ ]:
gene_variance_calib.groupby('Segment')[['ratio']].describe()

Compare distributions

In [ ]:


metrics = ["difference", "ratio"]

for m in metrics:
    cd = gene_variance_train.loc[
        gene_variance_train["Segment"]=="CD45+", m
    ].dropna()

    pk = gene_variance_train.loc[
        gene_variance_train["Segment"]=="PanCK+", m
    ].dropna()

    stat = mannwhitneyu(cd, pk, alternative="two-sided")

    print(f"\n{m}")
    print(f"CD45+ median: {cd.median():.4f}")
    print(f"PanCK+ median:{pk.median():.4f}")
    print(f"p-value: {stat.pvalue:.4f}")

In [ ]:
from scipy.stats import mannwhitneyu

metrics = ["difference", "ratio"]

for m in metrics:
    cd = gene_variance.loc[
        gene_variance["Segment"]=="CD45+", m
    ].dropna()

    pk = gene_variance.loc[
        gene_variance["Segment"]=="PanCK+", m
    ].dropna()

    stat = mannwhitneyu(cd, pk, alternative="two-sided")

    print(f"\n{m}")
    print(f"CD45+ median: {cd.median():.4f}")
    print(f"PanCK+ median:{pk.median():.4f}")
    print(f"p-value: {stat.pvalue:.4f}")

In [ ]:
from scipy.stats import mannwhitneyu

metrics = ["difference", "ratio"]

for m in metrics:
    cd = gene_variance_calib.loc[
        gene_variance_calib["Segment"]=="CD45+", m
    ].dropna()

    pk = gene_variance_calib.loc[
        gene_variance_calib["Segment"]=="PanCK+", m
    ].dropna()

    stat = mannwhitneyu(cd, pk, alternative="two-sided")

    print(f"\n{m}")
    print(f"CD45+ median: {cd.median():.4f}")
    print(f"PanCK+ median:{pk.median():.4f}")
    print(f"p-value: {stat.pvalue:.4f}")

In [ ]:
for m in ["measured_std", "predicted_std"]:
    cd = gene_variance_train.loc[gene_variance_train["Segment"]=="CD45+", m]
    pk = gene_variance_train.loc[gene_variance_train["Segment"]=="PanCK+", m]
    print(m, "CD45 median:", cd.median(), "PanCK median:", pk.median())

In [ ]:
combined_scores_unique

In [ ]:
for m in ["R2", "PearsonCorr"]:
    cd = combined_scores_unique.loc[combined_scores_unique["Segment"]=="CD45+", m]
    pk = combined_scores_unique.loc[combined_scores_unique["Segment"]=="PanCK+", m]
    print(m, "CD45 median:", cd.median(), "PanCK median:", pk.median())

### Plots

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = gene_variance  

# Consistent binning across segments
bins_diff  = np.linspace(df['difference'].min(), df['difference'].max(), 25)
bins_ratio = np.linspace(df['ratio'].min(),      df['ratio'].max(),      25)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# 1) Overlaid distributions of the paired difference by Segment
sns.histplot(
    data=df, x='difference', hue='Segment',
    bins=bins_diff, stat='density', common_norm=False, element='step',
    multiple='layer', ax=axes[0]
)
sns.kdeplot(data=df, x='difference', hue='Segment', common_norm=False, ax=axes[0], legend=False)
axes[0].axvline(0, ls='--', lw=1, c='k')
axes[0].set_xlabel('Measured SD − Predicted SD')
axes[0].set_title('Spread difference by Segment')

# 2) Overlaid distributions of the paired ratio by Segment
sns.histplot(
    data=df, x='ratio', hue='Segment',
    bins=bins_ratio, stat='density', common_norm=False,#element='poly',
    multiple='stack',
    ax=axes[1]
)
sns.kdeplot(data=df, x='ratio', hue='Segment', common_norm=False, ax=axes[1], legend=True)
axes[1].set_xlabel('Predicted SD / Measured SD')
axes[1].set_title('Spread ratio by Segment')

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = gene_variance_calib  

# Consistent binning across segments
bins_diff  = np.linspace(df['difference'].min(), df['difference'].max(), 25)
bins_ratio = np.linspace(df['ratio'].min(),      df['ratio'].max(),      25)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))



# 1) Overlaid distributions of the paired ratio by Segment

gv = gene_variance_calib.copy()
long = gv.rename(columns={'measured_std':'Measured','predicted_std':'Predicted'})\
         .melt(value_name='SD', var_name='Source', value_vars=['Measured','Predicted'])

# shared bins across both groups
lo = long['SD'].min(); hi = long['SD'].max()
bins = np.linspace(lo, hi, 25)

sns.histplot(data=long, x='SD', hue='Source', bins=bins,
             stat='density', multiple='layer', element='step', common_norm=False, ax=axes[0])
sns.kdeplot(data=long, x='SD', hue='Source', common_norm=False, ax=axes[0])

for m,lab in [(gv.measured_std.mean(),'Measured'), (gv.predicted_std.mean(),'Predicted')]:
    axes[0].axvline(m, ls='--', lw=1)
    
axes[0].set_xlabel('Per-gene SD'); 
axes[0].set_ylabel('Density'); 
axes[0].set_title('Spread of variance across genes - calibrated')

# 1) Overlaid distributions of the paired difference by Segment
sns.histplot(
    data=df, x='difference', hue='Segment',
    bins=bins_diff, stat='density', common_norm=False, element='step',
    multiple='layer', ax=axes[1]
)
sns.kdeplot(data=df, x='difference', hue='Segment', common_norm=False, ax=axes[1], legend=False)
axes[1].axvline(0, ls='--', lw=1, c='k')
axes[1].set_xlabel('Measured SD − Predicted SD')
axes[1].set_title('Spread difference by Segment - calibrated')
axes[1].set_xlim(-1.5,1.5)

# axes[0].text(-0.1, 1.05, 'A', transform=axes[0].transAxes,
#              fontsize=14, fontweight='bold', va='top', ha='center')

# axes[1].text(-0.1, 1.05, 'B', transform=axes[1].transAxes,
#              fontsize=14, fontweight='bold', va='top', ha='center')

plt.tight_layout()
plt.show()



In [ ]:
plt.figure(figsize=(6,4))
sns.ecdfplot(data=long, x='SD', hue='Source')
plt.xlabel('Per-gene SD'); plt.ylabel('ECDF')
plt.xlim(0,3.5)
plt.title('ECDF: Measured vs Predicted SD')
plt.tight_layout()

In [ ]:
def safe_linspace_bins(x, n=25):
    x = np.asarray(x)
    x = x[np.isfinite(x)]
    if x.size < 2:
        return None
    lo, hi = x.min(), x.max()
    if lo == hi:
        # expand a tiny bit so bins have nonzero width
        eps = 1e-6 if lo == 0 else abs(lo) * 1e-6
        lo, hi = lo - eps, hi + eps
    return np.linspace(lo, hi, n)

# VR plot def

In [ ]:
def plot_variance_row(df, axes,field='difference', field_goal=0,label="", show_titles=False,xlim=(-2,2)):

    field_min = df[field].min()
    field_max = df[field].max()
    # Consistent binning across segments
    bins_diff = np.linspace(field_min, field_max, 25)
    bins_ms   = np.linspace(df['measured_std'].min(), df['measured_std'].max(), 25)


    gv = df.copy()
    long = gv.rename(columns={'measured_std':'Measured','predicted_std':'Predicted'})\
            .melt(value_name='SD', var_name='Source', value_vars=['Measured','Predicted'])

    # shared bins across both groups
    lo = long['SD'].min(); hi = long['SD'].max()
    bins = np.linspace(lo, hi, 25)

    palette = {
        'Measured':  "#40B0A6",  # 
        'Predicted': "#5D3A9B",  # 
    }

    sns.histplot(data=long, x='SD', hue='Source', bins=bins, palette=palette,
                stat='density', multiple='layer', element='step', common_norm=False, ax=axes[0])
    sns.kdeplot(data=long, x='SD', hue='Source', palette=palette, common_norm=False, ax=axes[0])

    axes[0].axvline(gv.measured_std.mean(), ls='--', lw=1, c="#40B0A6")
    axes[0].axvline(gv.predicted_std.mean(), ls='--', lw=1, c="#5D3A9B")
        
    axes[0].set_xlabel('Per-gene variance')#, fontsize=8)
    axes[0].set_ylabel(None)
    axes[0].set_yticks([])
    axes[0].set_xlim(-0.25,4.0)
    if show_titles: axes[0].set_title(f'Distribution of Variance')#, fontsize=10)

    # 1) field
    sns.histplot(
        data=df, x=field, color="#5D3A9B",
        bins=bins_diff, stat='density', common_norm=False, element='step',
        multiple='layer', ax=axes[1]
    )
    sns.kdeplot(data=df, x=field, common_norm=False, color="#5D3A9B",ax=axes[1], legend=False)
    axes[1].axvline(field_goal, ls='--', lw=1, c='grey')
    mean_val = df[field].mean()

    axes[1].axvline(mean_val, ls='--', lw=1, c="#5D3A9B")
    axes[1].text(
        0.98, 0.95,                      # top-right corner of the axes
        f"mean = {mean_val:.3f}",
        transform=axes[1].transAxes,     # interpret (x, y) in axes coords
        ha='right',
        va='top',
        fontsize=8,
    )
    axes[1].set_xlabel('Variance Ratio')#, fontsize=8)
    axes[1].set_ylabel(None)
    axes[1].set_yticks([])
    axes[1].set_xlim(xlim)
    if show_titles: axes[1].set_title(f'Variance Ratio of dataset')#, fontsize=10)

    # 2) field by Segment
    sns.histplot(
        data=df, x=field, hue='Segment',
        bins=bins_diff, stat='density', common_norm=False, element='step',
        multiple='layer', ax=axes[2]
    )
    sns.kdeplot(data=df, x=field, hue='Segment', common_norm=False, ax=axes[2], legend=False)
    axes[2].axvline(field_goal, ls='--', lw=1, c='grey')
    axes[2].axvline(df.loc[df['Segment']=='CD45+',field].mean(), ls='--', lw=1, c="#6FD9EE")
    axes[2].axvline(df.loc[df['Segment']=='PanCK+',field].mean(), ls='--', lw=1, c="#F48D1F")
    axes[2].set_xlabel('Variance Ratio')#, fontsize=8)
    axes[2].set_ylabel('')
    axes[2].set_xlim(xlim)
    if show_titles: axes[2].set_title(f'Variance Ratio by Compartment')#, fontsize=10)

    if len(axes)>3:
        # 2) Measured std by Segment
        sns.histplot(
            data=df, x='measured_std', hue='Segment',
            bins=bins_ms, stat='density', common_norm=False, element='step',
            multiple='layer', ax=axes[3]
        )
        sns.kdeplot(data=df, x='measured_std', hue='Segment', common_norm=False, ax=axes[2], legend=False)
        axes[3].set_xlabel('Measured SD')#, fontsize=8)
        axes[3].set_ylabel('')
        axes[3].set_xlim(0,4)
        if show_titles: axes[3].set_title(f'{label}: Ground Truth Variance', fontsize=10)

### put an annotaiton on axes[0] for the label


In [ ]:
gene_variance_calib.to_csv(os.path.join(save_path,'genevar_calib.csv'))

#### Supp Fig - Variance Ratio

In [ ]:
# RATIO
fig, axes = plt.subplots(3, 3, figsize=(6.5, 6))
xlim=(0,2)
plot_variance_row(df=gene_variance_train, axes=axes[0,:], field='ratio', field_goal=1 ,xlim=xlim, label="Train")
plot_variance_row(df=gene_variance, axes=axes[1,:], field='ratio', field_goal=1 ,xlim=xlim,label="Test - Raw")
plot_variance_row(df=gene_variance_calib, axes=axes[2,:], field='ratio', field_goal=1 ,xlim=xlim,label="Test - Calibrated")


axes[0,0].text(
    -0.2, 1.05,
    "A",
    transform=axes[0,0].transAxes,
    fontsize=10,
    fontweight='bold'
)

axes[0,0].text(
    -0.1, 0.5,
    "Train dataset",
    transform=axes[0,0].transAxes,
    fontsize=8,
    ha='center', va='center', rotation=90
)



axes[1,0].text(
    -0.2, 1.05,
    "B",
    transform=axes[1,0].transAxes,
    fontsize=10,
    fontweight='bold'
)

axes[1,0].text(
    -0.1, 0.5,
    "Test dataset - Uncalibrated",
    transform=axes[1,0].transAxes,
    fontsize=8,
    ha='center', va='center', rotation=90
)



axes[2,0].text(
    -0.2, 1.05,
    "C",
    transform=axes[2,0].transAxes,
    fontsize=10,
    fontweight='bold'
)

axes[2,0].text(
    -0.1, 0.5,
    "Test dataset - Calibrated",
    transform=axes[2,0].transAxes,
    fontsize=8,
    ha='center', va='center', rotation=90
)


#plt.suptitle(f'Calibration Set {CALIBRATION_FRACTION:.0%}')
plt.tight_layout()
plt.savefig(os.path.join(fig_path,"suppfig-vr.pdf"), format='pdf', bbox_inches='tight')  
plt.show()

In [ ]:
#DIFFERENCE
fig, axes = plt.subplots(3, 3, figsize=(10, 9))

plot_variance_row(df=gene_variance_train, axes=axes[0,:] ,label="Train")
plot_variance_row(df=gene_variance, axes=axes[1,:] ,label="Test")
plot_variance_row(df=gene_variance_calib, axes=axes[2,:] ,label="Calibrated")


plt.suptitle(f'Calibration Set {CALIBRATION_FRACTION:.0%}')
plt.tight_layout()
plt.show()


In [ ]:
#DIFFERENCE
fig, axes = plt.subplots(2, 3, figsize=(10, 6))


plot_variance_row(df=gene_variance_calib, axes=axes[0,:], field='ratio', field_goal=1 ,xlim=(0,2),label="Calibrated")
plot_variance_row(df=gene_variance_calib, axes=axes[1,:] ,label="Calibrated")


plt.suptitle(f'Calibration Set {CALIBRATION_FRACTION:.0%}')
plt.tight_layout()
plt.show()


## Plots

In [ ]:
from scipy.stats import combine_pvalues


In [ ]:
def fisher_with_stat(pvals):
    stat, pval = combine_pvalues(pvals, method='fisher')
    return pd.Series({'FisherStat': stat, 'CombinedPval': pval})


In [ ]:
seg_test_gene_scores['CI_width'] = seg_test_gene_scores['PearsonCI_high']-seg_test_gene_scores['PearsonCI_low']

In [ ]:
print(f"captured varience: {(geneset_good_models.shape[0] / combined_scores_unique.shape[0]):.2%}")

In [ ]:
geneset_good_models['CI_width'] = geneset_good_models['PearsonCI_high']-geneset_good_models['PearsonCI_low']
geneset_good_models.groupby(['Segment','GeneSet'])[['PearsonCorr','R2','PearsonQVals','CI_width']].mean()

In [ ]:
seg_test_gene_scores.groupby(['Segment','GeneSet'])[['PearsonCorr','R2','PearsonQVals','CI_width']].mean()

In [ ]:
geneset_good_models.groupby(['Segment','GeneSet'])[['PearsonCI_high','PearsonCI_low']].max()

In [ ]:
seg_test_gene_scores.groupby(['Segment','GeneSet'])[['PearsonCI_high','PearsonCI_low']].max()

In [ ]:
(seg_test_gene_scores
    .groupby(['Segment','GeneSet'])['PearsonPVals']
    .apply(lambda pvals: combine_pvalues(pvals.values, method='fisher')[0])
    .reset_index()
    .rename(columns={'PearsonPVals':'FisherStat'})
)

Proportion of genes that are highly significantly correlated (pval < 0.001)

In [ ]:
sig_prop = seg_test_gene_scores.assign(Significant = seg_test_gene_scores['PearsonPVals'] < 0.001).groupby(['Segment', 'GeneSet'])['Significant'].mean()
sig_prop

## Analyse Results

In [ ]:
geneset_good_models

In [ ]:
#CALIBRATED
# Set up the FacetGrid to split by 'Segment'
g = sns.catplot(
    data=geneset_good_models,
    x='Segment',
    y='ConcordanceCorr_calib',
    hue='GeneSet',
    kind='box',
    height=5,
    aspect=1.2,
    order=segment_order,
    hue_order=geneset_order,
    palette='viridis'
)

# Add titles and formatting
g.set_titles("Segment: {col_name}")
g.set_axis_labels("", "Concordance Correlation Coefficient")
#g.set(ylim=(0.5, 1.2))  # Set y-axis range
g._legend.set_bbox_to_anchor((0.95,0.9))
g._legend.set_loc('upper right')
g._legend.set_frame_on(True)  

new_labels = ['Cancer Panel', 'Highly Variable Genes','Spatially Variable Genes']
for t, l in zip(g._legend.texts, new_labels):
    t.set_text(l)

g.ax.set_xticklabels(["Immune Regions (CD45+)", "Tumour Regions (PanCK+)"])

plt.suptitle("Correlation between predicted and measured expression in strong predictors only")
plt.tight_layout()
plt.show()

In [ ]:


# Set up the FacetGrid to split by 'Segment'
g = sns.catplot(
    data=geneset_good_models,
    x='Segment',
    y='PearsonCorr',
    hue='GeneSet',
    kind='box',
    height=5,
    aspect=1.2,
    order=segment_order,
    hue_order=geneset_order,
    palette='viridis'
)

# Add titles and formatting
g.set_titles("Segment: {col_name}")
g.set_axis_labels("", "Pearson Correlation Coefficient (ρ)")
#g.set(ylim=(0.5, 1.2))  # Set y-axis range
g._legend.set_bbox_to_anchor((0.95,0.9))
g._legend.set_loc('upper right')
g._legend.set_frame_on(True)  

new_labels = ['Cancer Panel', 'Highly Variable Genes','Spatially Variable Genes']
for t, l in zip(g._legend.texts, new_labels):
    t.set_text(l)

g.ax.set_xticklabels(["Immune Regions (CD45+)", "Tumour Regions (PanCK+)"])

plt.suptitle("Correlation between predicted and measured expression in strong predictors only")
plt.tight_layout()
plt.show()

## Compare scores

Examine how many unique gene-models captured a large amount of the variance. We are only using the results from the combined features (rather than Virchow/UNI) 

In [ ]:
fm_cond = seg_test_gene_scores['Feature_Extractor']=='fused'

## Stats for results

In [ ]:
print("unique genes:")
print(f"R2 > 0.2: {geneset_good_models[geneset_good_models['R2']>0.2]['Gene'].shape[0]}")
print(f"R2 > 0.3: {geneset_good_models[geneset_good_models['R2']>0.3]['Gene'].shape[0]}")
print(f"R2 > 0.4: {geneset_good_models[geneset_good_models['R2']>0.4]['Gene'].shape[0]}")

unique genes:
R2 > 0.2: 147
R2 > 0.3: 102
R2 > 0.4: 69

In [ ]:
geneset_good_models.columns

In [ ]:
cols = ['nRMSE','R2','PearsonCorr','ConcordanceCorr','ConcordanceCorr_calib']
geneset_good_models[fm_cond].groupby('Segment')[cols].aggregate('median')

	nRMSE	R2	PearsonCorr	ConcordanceCorr	ConcordanceCorr_calib
Segment					
CD45+	0.707	0.496	0.731	0.661	0.704
PanCK+	0.845	0.281	0.573	0.485	0.469

## Examine Gene-Level predictions

In [ ]:
cols = ['Gene','R2','PearsonCorr','PearsonQVals','PearsonCI_low','PearsonCI_high','ConcordanceCorr','ConcordanceCorr_calib', 'R2_calib']
cond = (combined_scores_unique['Gene']=='MKI67')&(combined_scores_unique['Feature_Extractor']=='fused')
combined_scores_unique.loc[cond,cols]

## Correlation

### Top 10 genes by segment

In [ ]:
geneset_good_models = geneset_good_models[geneset_good_models['Feature_Extractor']=='fused']


In [ ]:
num_genes = 10
df = geneset_good_models.copy()


# Select top 10 genes by Pearson_r (or use q-value filter if preferred)
top_genes_panck = df[df['Segment']=='PanCK+'].nlargest(num_genes, "ConcordanceCorr_calib").copy()
top_genes_cd45 = df[df['Segment']=='CD45+'].nlargest(num_genes, "ConcordanceCorr_calib").copy()

# Sort for visual clarity
top_genes_panck = top_genes_panck.sort_values("ConcordanceCorr_calib", ascending=True)
top_genes_cd45 = top_genes_cd45.sort_values("ConcordanceCorr_calib", ascending=True)


fig, axes = plt.subplots(1, 2, figsize=(10, 5),gridspec_kw={'wspace': 0.3} ) 


axes[0].barh(
    top_genes_cd45["Gene"],
    top_genes_cd45["ConcordanceCorr_calib"],
    #xerr=[top_genes_cd45["ci_lower_error"], top_genes_cd45["ci_upper_error"]],
    align='center',
    ecolor='black',
    capsize=5,
    color='skyblue'
)

for i, (val, qval) in enumerate(zip(
    top_genes_cd45["ConcordanceCorr_calib"],
    top_genes_cd45["PearsonQVals"]
)):
    if qval <= 0.001:
        sig = "***"
    elif qval <= 0.01:
        sig = "**"
    elif qval <= 0.05:
        sig = "*"
    else:
        sig = ""
    xpos = val + 0.01  # add small offset to avoid overlap
    axes[0].text(xpos, i, sig, va='center', fontsize=10, fontweight='bold')

axes[0].set_xlabel("Concordance Correlation Coefficient")
axes[0].grid(axis='x', linestyle='--', alpha=0.7)
axes[0].set_title('Immune Regions (CD45⁺)', fontsize=14)
axes[0].set_xlim((0,1.0))


axes[1].barh(
    top_genes_panck["Gene"],
    top_genes_panck["ConcordanceCorr_calib"],
    #xerr=[top_genes_panck["ci_lower_error"], top_genes_panck["ci_upper_error"]],
    align='center',
    ecolor='black',
    capsize=5,
    color='skyblue'
)
for i, (val, qval) in enumerate(zip(
    top_genes_panck["ConcordanceCorr_calib"],
    top_genes_panck["PearsonQVals"]
)):
    if qval <= 0.001:
        sig = "***"
    elif qval <= 0.01:
        sig = "**"
    elif qval <= 0.05:
        sig = "*"
    else:
        sig = ""
    xpos = val  + 0.01  # add small offset to avoid overlap
    axes[1].text(xpos, i, sig, va='center', fontsize=10, fontweight='bold')

axes[1].set_xlabel("Concordance Correlation Coefficient")
axes[1].grid(axis='x', linestyle='--', alpha=0.7)
axes[1].set_title('Tumour Regions (PanCK⁺)', fontsize=14)
axes[1].set_xlim((0,1.0))


plt.suptitle(f"Top {num_genes} genes by Concordance Correlation Coefficient", fontsize=20)
plt.figtext(0.5, -0.05, '*** indicates p < 0.001', ha='center', fontsize=14)

plt.show()


In [ ]:
# Select top 10 genes by Pearson_r (or use q-value filter if preferred)
num_genes=20
top_genes_panck = df[df['Segment']=='PanCK+'].nlargest(num_genes, "ConcordanceCorr_calib").copy()
top_genes_cd45 = df[df['Segment']=='CD45+'].nlargest(num_genes, "ConcordanceCorr_calib").copy()

# Sort for visual clarity
top_genes_panck = top_genes_panck.sort_values("ConcordanceCorr_calib", ascending=False)
top_genes_cd45 = top_genes_cd45.sort_values("ConcordanceCorr_calib", ascending=False)

In [ ]:

num_genes = 20
df = geneset_good_models.copy()
df["ci_lower_error"] = df["PearsonCorr"] - df["PearsonCI_low"]
df["ci_upper_error"] = df["PearsonCI_high"] - df["PearsonCorr"]
df["ci_lower_error"] = df["ci_lower_error"].clip(lower=0)
df["ci_upper_error"] = df["ci_upper_error"].clip(lower=0)



# Select top 10 genes by Pearson_r (or use q-value filter if preferred)
top_genes_panck = df[df['Segment']=='PanCK+'].nlargest(num_genes, "PearsonCorr").copy()
top_genes_cd45 = df[df['Segment']=='CD45+'].nlargest(num_genes, "PearsonCorr").copy()

# Sort for visual clarity
top_genes_panck = top_genes_panck.sort_values("PearsonCorr", ascending=True)
top_genes_cd45 = top_genes_cd45.sort_values("PearsonCorr", ascending=True)



## Supp Figure - Correlation

In [ ]:
top_genes_cd45[['Gene','PearsonCorr','PearsonQVals']].sort_values('PearsonCorr', ascending=False)[:2]

In [ ]:
top_genes_panck[['Gene','PearsonCorr','PearsonQVals']].sort_values('PearsonCorr', ascending=False)[:2]

In [ ]:
combined_scores_unique

In [ ]:
mosaic =    """
            AB
            CD
            """

fig, axes = plt.subplot_mosaic(mosaic, figsize=(7, 9), #12,10
                               gridspec_kw={'wspace':.1,'hspace':0.1,
                                            'height_ratios': [1, 2]  },
                               constrained_layout=True)
#fig, axes = plt.subplots( 1, 2, figsize=(12,5))

### A

sns.barplot(
    data=combined_scores_unique,
    x='Segment',
    y='PearsonCorr',
    hue='GeneSet',
    order=segment_order,
    hue_order=geneset_order,
    palette='viridis',
    errorbar='sd',
    ax=axes['A']
)
# Add value labels automatically
for container in axes['A'].containers:
    axes['A'].bar_label(container, fmt="%.2f", label_type='center',  color='white')

# Replace x-tick labels
axes['A'].set_xticks(range(len(segment_order)))  # Optional but useful

axes['A'].set_xticklabels(["Immune (CD45+)", "Tumour (PanCK+)"])#, fontsize=LABELSIZE)

# Axis labels and title
axes['A'].set_ylabel("Pearson's ρ")#, fontsize=LABELSIZE) #, fontsize=12)
axes['A'].set_xlabel("")
axes['A'].set_ylim(0.0, 1.0)
#axes[0].set_title("Correlation between predicted and measured expression in hold-out test set")

# Legend formatting
handles, labels = axes['A'].get_legend_handles_labels()
new_labels = ['Cancer Panel', 'Highly Variable Genes', 'Spatially Variable Genes']
axes['A'].legend(handles, new_labels, title="GeneSet", loc='upper right', frameon=True)


### B
sns.histplot(
    data=combined_scores_unique,
    x='PearsonCorr',
    hue='Segment',
    bins=20,
    element='bars',  # or "bars"
    common_norm=False,  # don’t normalize across groups
    palette='viridis',
    edgecolor='black',
    ax=axes['B']
)
#axes[1].set_title("Distribution of correlation between measured and predicted expression")
axes['B'].set_xlabel("Pearson's ρ")#, fontsize=LABELSIZE)
axes['B'].set_ylabel("Count")#, fontsize=LABELSIZE)

## cant use get_legend_handles... for some reason
legend = axes['B'].get_legend()
if legend:
    labels = ['Immune (CD45+)', 'Tumour (PanCK+)']
    for text, new_label in zip(legend.get_texts(), labels):
        text.set_text(new_label)



###C



axes['C'].barh(
    top_genes_cd45["Gene"],
    top_genes_cd45["PearsonCorr"],
    xerr=[top_genes_cd45["ci_lower_error"], top_genes_cd45["ci_upper_error"]],
    align='center',
    ecolor='black',
    capsize=5,
    color='skyblue'
)

for i, (val, upper_err, qval) in enumerate(zip(
    top_genes_cd45["PearsonCorr"],
    top_genes_cd45["ci_upper_error"],
    top_genes_cd45["PearsonQVals"]
)):
    if qval <= 0.001:
        sig = "***"
    elif qval <= 0.01:
        sig = "**"
    elif qval <= 0.05:
        sig = "*"
    else:
        sig = ""
    xpos = val + upper_err + 0.01  # add small offset to avoid overlap
    axes['C'].text(xpos, i, sig, va='center', fontsize=10, fontweight='bold')

axes['C'].set_xlabel("Pearson's ρ")#, fontsize=LABELSIZE)
axes['C'].grid(axis='x', linestyle='--', alpha=0.7)
axes['C'].set_title('Immune Regions (CD45+)')#, fontsize=LABELSIZE)
axes['C'].set_xlim((0,1.0))

### D

axes['D'].barh(
    top_genes_panck["Gene"],
    top_genes_panck["PearsonCorr"],
    xerr=[top_genes_panck["ci_lower_error"], top_genes_panck["ci_upper_error"]],
    align='center',
    ecolor='black',
    capsize=5,
    color='skyblue'
)
for i, (val, upper_err, qval) in enumerate(zip(
    top_genes_panck["PearsonCorr"],
    top_genes_panck["ci_upper_error"],
    top_genes_panck["PearsonQVals"]
)):
    if qval <= 0.001:
        sig = "***"
    elif qval <= 0.01:
        sig = "**"
    elif qval <= 0.05:
        sig = "*"
    else:
        sig = ""
    xpos = val + upper_err + 0.01  # add small offset to avoid overlap
    axes['D'].text(xpos, i, sig, va='center', fontsize=10, fontweight='bold')

axes['D'].set_xlabel("Pearson's ρ")#, fontsize=LABELSIZE)
axes['D'].grid(axis='x', linestyle='--', alpha=0.7)
axes['D'].set_title('Tumour Regions (PanCK+)') #, fontsize=TITLESIZE)
axes['D'].set_xlim((0,1.0))



#### ALL

#fig.suptitle('Correlation between measured and predicted expression', ha='center', fontsize=SUPTITLESIZE)
plt.figtext(0.5, -0.025, '* p < 0.05 | ** p < 0.01 | *** p < 0.001', ha='center') #, fontsize=NOTESIZE)

### Prepare for paper
#plt.tight_layout()
#plt.subplots_adjust(wspace=0.3)
axes['A'].text(-0.1, 1.05, 'A', transform=axes['A'].transAxes,
              fontweight='bold', va='top', ha='center') #fontsize=14,

axes['B'].text(-0.1, 1.05, 'B', transform=axes['B'].transAxes,
              fontweight='bold', va='top', ha='center') #fontsize=14,

axes['C'].text(-0.1, 1.05, 'C', transform=axes['C'].transAxes,
              fontweight='bold', va='top', ha='center') #fontsize=14,

plt.savefig(os.path.join(fig_path,"suppfig-pcc.pdf"), format='pdf', bbox_inches='tight')  


plt.show()



# Vizualise expression

In [ ]:
import matplotlib as mpl
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from matplotlib.patches import Rectangle
from matplotlib.transforms import Bbox
import matplotlib.patches as patches
from matplotlib.cm import ScalarMappable
from skimage.transform import resize

In [ ]:
from LEAPdata import get_slide_id
from visualize import plot_rois_on_image

from PIL import Image
import PIL

# Increase the image pixel limit
PIL.Image.MAX_IMAGE_PIXELS = None  

### Viz functions

In [ ]:
from matplotlib.ticker import MaxNLocator, FormatStrFormatter

In [ ]:
def plot_patient_row(fig, gs_row, image_gt, image_pred, values_gt, values_pred,
                     rois, sample_metadata, leap_id, limits, cmap='seismic',
                     show_titles=False, ds=0.2):
    rect_w = 224*ds
    vmin = min(values_gt.min(), values_pred.min())
    vmax = max(values_gt.max(), values_pred.max())+0.5

    cmap = mpl.colormaps[cmap]  if isinstance(cmap, str) else cmap
    norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)
    sm = ScalarMappable(norm=norm, cmap=cmap)

    # Left label
    ax_label = fig.add_subplot(gs_row[0])
    ax_label.axis('off')
    ax_label.text(0.95, 0.5, leap_id, fontsize=18, ha='right', va='center', transform=ax_label.transAxes,rotation=45 )

    # Ground Truth
    ax_gt = fig.add_subplot(gs_row[1])
    ax_gt.imshow(image_gt)
    for x, y, val in zip(sample_metadata.loc[rois, "scatter_x"],
                         sample_metadata.loc[rois, "scatter_y"],
                         values_gt):
        
        color = cmap(norm(val)) 
        rect = patches.Rectangle((x - rect_w/2, y - rect_w/2), rect_w, rect_w, facecolor=color, edgecolor=None, alpha=0.95)
        ax_gt.add_patch(rect)
    if show_titles:
        ax_gt.set_title("Measured", fontsize=TITLESIZE)
    ax_gt.set_xlim(limits[0])
    ax_gt.set_ylim(limits[1])
    ax_gt.axis('off')
    ax_gt.set_aspect('auto')
    # Prediction
    
    # Prediction
    ax_pred = fig.add_subplot(gs_row[2])
    ax_pred.imshow(image_pred)
    for x, y, val in zip(sample_metadata.loc[rois, "scatter_x"],
                         sample_metadata.loc[rois, "scatter_y"],
                         values_pred):
        color = cmap(norm(val))
        rect = patches.Rectangle((x - rect_w/2, y - rect_w/2), rect_w, rect_w, facecolor=color, edgecolor=None, alpha=0.8)
        ax_pred.add_patch(rect)
    if show_titles:
        ax_pred.set_title("Predicted", fontsize=18)
    ax_pred.set_xlim(limits[0])
    ax_pred.set_ylim(limits[1])
    ax_pred.axis('off')
    ax_pred.set_aspect('auto')

    # Colorbar
    cbar_ax = inset_axes(
        ax_pred,           # attach to the predicted image plot
        width="3%",        # % of parent width
        height="100%",     # match height of ax_pred
        loc='center left',
        bbox_to_anchor=(1.02, 0., 1, 1),
        bbox_transform=ax_pred.transAxes,
        borderpad=0
    )
    cb = plt.colorbar(sm, cax=cbar_ax)
    cb.set_label("Expression Level", fontsize=14)
    # 1) force integer tick *locations*
    cb.locator = MaxNLocator(integer=True)  
    # 2) force integer tick *formatting*
    cb.formatter = FormatStrFormatter("%d")
    cb.update_ticks()


In [ ]:
def plot_roiexpr_on_he(he_img, y_test, y_pred, rois, gene, subset_metadata, xlim=None, ylim=None,ds=1.0):
   
    gene_cols = y_test.columns.drop("LEAP_ID") if "LEAP_ID" in y_test.columns else y_test.columns
    rect_w = 224 * ds

    y_test = y_test.loc[rois,:]
    y_pred = y_pred.loc[rois,:]
    subset_metadata = subset_metadata.loc[rois,:]
    #print(subset_metadata['scatter_x'])

    vmin = min(y_test[gene].min(), y_pred[gene].min())
    vmax = max(y_test[gene].max(), y_pred[gene].max())
    cmap = plt.cm.seismic #bwr #autumn #plt.cm.hot  # or 'viridis', etc.

    fig, axes = plt.subplots(1, 2, figsize=(14, 5)) 

    # Shared colormap normalizer
    norm = plt.Normalize(vmin=vmin, vmax=vmax)
    sm = ScalarMappable(cmap=cmap, norm=norm)

    for ax, values, title in zip(axes, [y_test[gene], y_pred[gene]],
                                [f"Ground Truth Expression of {gene}",
                                f"Predicted Expression of {gene}"]):

        ax.imshow(he_img)
        
        for x, y, val in zip(
            subset_metadata["scatter_x"],
            subset_metadata["scatter_y"],
            values
        ):
            color = cmap(norm(val))
            rect = patches.Rectangle(
                (x - rect_w/2, y - rect_w/2), rect_w, rect_w,  # (left, bottom), width, height
                linewidth=0,
                edgecolor=None,
                facecolor=color,
                alpha=0.8
            )
            ax.add_patch(rect)

        ax.set_title(title)
        ax.set_xlabel("X Coordinate")
        ax.set_ylabel("Y Coordinate")
        if ylim: ax.set_ylim(ylim)
        if xlim: ax.set_xlim(xlim)

    # Add colorbars
    for ax in axes:
        plt.colorbar(sm, ax=ax, label="Expression Level", shrink=0.75)

    plt.suptitle(f'Actual vs Predicted Gene Expression - {segment} {gene}')
    plt.tight_layout()#
    
    plt.show()


In [ ]:
def plot_roirank_on_he(he_img, y_test, y_pred, rois, gene, subset_metadata, xlim=None, ylim=None,ds=1.0):
   
    gene_cols = y_test.columns.drop("LEAP_ID") if "LEAP_ID" in y_test.columns else y_test.columns
    rect_w = 224 * ds


    #vmin = min(y_test[gene].min(), y_pred[gene].min())
    #vmax = max(y_test[gene].max(), y_pred[gene].max())
    #print(y_test.loc[rois,:])
    y_test = y_test.loc[rois,:]
    y_pred = y_pred.loc[rois,:]
    subset_metadata = subset_metadata.loc[rois,:]

    cmap = plt.cm.seismic #bwr #autumn #plt.cm.hot  # or 'viridis', etc.

    fig, axes = plt.subplots(1, 2, figsize=(14, 5)) 

    
    #print(y_test[gene])
    for ax, values, title in zip(axes, [y_test[gene], y_pred[gene]],
                                [f"Ground Truth Expression of {gene}",
                                f"Predicted Expression of {gene}"]):
        # Independent normalization for this axis
        vmin, vmax = values.min(), values.max()
        #print(f"v: {vmin} - {vmax}")
        norm = plt.Normalize(vmin=vmin, vmax=vmax)
        sm = ScalarMappable(cmap=cmap, norm=norm)
        sm.set_array([])  # Required for colorbar

        ax.imshow(he_img)
        
        for x, y, val in zip(
            subset_metadata["scatter_x"],
            subset_metadata["scatter_y"],
            values
        ):
            color =  plt.cm.seismic((val - vmin) / (vmax - vmin)) 
            rect = patches.Rectangle(
                (x - rect_w/2, y - rect_w/2), rect_w, rect_w,  # (left, bottom), width, height
                linewidth=0,
                edgecolor=None,
                facecolor=color,
                alpha=0.8
            )
            ax.add_patch(rect)

        ax.set_title(title)
        ax.set_xlabel("X Coordinate")
        ax.set_ylabel("Y Coordinate")
        if ylim: ax.set_ylim(ylim)
        if xlim: ax.set_xlim(xlim)

    # Add colorbars
    #for ax in axes:
    #    plt.colorbar(sm, ax=ax, label="Expression Level", shrink=0.75)

    sm = plt.cm.ScalarMappable(cmap='seismic', norm=plt.Normalize(vmin=vmin, vmax=vmax))
    fig.colorbar(sm, ax=ax, label="Expression Level", shrink=0.75)
    plt.suptitle(f'Actual vs Predicted Gene Expression - {segment} {gene}')
    plt.tight_layout()#
    
    plt.show()

In [ ]:
def plot_expr_on_he_both(he_img, y_test, y_pred, rois, gene, subset_metadata, xlim=None, ylim=None):
   
    gene_cols = y_test.columns.drop("LEAP_ID") if "LEAP_ID" in y_test.columns else y_test.columns


    y_test = y_test.loc[rois,:]
    y_pred = y_pred.loc[rois,:]
    subset_metadata = subset_metadata.loc[rois,:]


    vmin1 = min(y_test[gene].min(), y_pred[gene].min())
    vmax1 = max(y_test[gene].max(), y_pred[gene].max())
    vmin2 = vmin1
    vmax2 = vmax1


    fig, axes = plt.subplots(2, 2, figsize=(14, 10)) 
    axes = axes.flatten()

    ## INDIVIDUAL SCALES
    # Plot 1: Ground Truth
    sc1 = axes[0].imshow(he_img)
    scatter1 = axes[0].scatter(
        subset_metadata['scatter_x'], 
        subset_metadata['scatter_y'], 
        c=y_test[gene], cmap="seismic", alpha=1
    )
    axes[0].set_title(f"Ground Truth Expression of {gene}")
    axes[0].set_xlabel("X Coordinate")
    axes[0].set_ylabel("Y Coordinate")
    if ylim: axes[0].set_ylim(ylim)
    if xlim: axes[0].set_xlim(xlim)
    plt.colorbar(scatter1, ax=axes[0], label="Expression Level",shrink=0.75)

    # Plot 2: Prediction
    sc2 = axes[1].imshow(he_img)
    scatter2 = axes[1].scatter(
        subset_metadata['scatter_x'], 
        subset_metadata['scatter_y'], 
        c=y_pred[gene], cmap="seismic", alpha=1
    )
    axes[1].set_title(f"Predicted Expression of {gene}")
    axes[1].set_xlabel("X Coordinate")
    axes[1].set_ylabel("Y Coordinate")
    if ylim: axes[1].set_ylim(ylim)
    if xlim: axes[1].set_xlim(xlim)
    plt.colorbar(scatter2, ax=axes[1], label="Expression Level",shrink=0.75)


    ## SHARED SCALES
    # Plot 1: Ground Truth
    sc1 = axes[2].imshow(he_img)
    scatter1 = axes[2].scatter(
        subset_metadata['scatter_x'], 
        subset_metadata['scatter_y'], 
        c=y_test[gene], cmap="seismic", alpha=1, vmin=vmin1,vmax=vmax1
    )
    axes[2].set_title(f"Ground Truth Expression of {gene}")
    axes[2].set_xlabel("X Coordinate")
    axes[2].set_ylabel("Y Coordinate")
    if ylim: axes[2].set_ylim(ylim)
    if xlim: axes[2].set_xlim(xlim)
    plt.colorbar(scatter1, ax=axes[2], label="Expression Level",shrink=0.75)

    # Plot 2: Prediction
    sc2 = axes[3].imshow(he_img)
    scatter2 = axes[3].scatter(
        subset_metadata['scatter_x'], 
        subset_metadata['scatter_y'], 
        c=y_pred[gene], cmap="seismic", alpha=1, vmin=vmin2,vmax=vmax2
    )
    axes[3].set_title(f"Predicted Expression of {gene}")
    axes[3].set_xlabel("X Coordinate")
    axes[3].set_ylabel("Y Coordinate")
    if ylim: axes[3].set_ylim(ylim)
    if xlim: axes[3].set_xlim(xlim)
    plt.colorbar(scatter2, ax=axes[3], label="Expression Level",shrink=0.75)



    plt.suptitle(f'Actual vs Predicted Gene Expression - {segment} {gene}')
    plt.tight_layout()#
    
    plt.show()

In [ ]:
def plot_expr_on_he(he_img, y_test, y_pred_np, rois, gene, subset_metadata, xlim=None, ylim=None,downsample_factor=0.2, same_scale=True):
   
    gene_cols = y_test.columns.drop("LEAP_ID") if "LEAP_ID" in y_test.columns else y_test.columns

    y_pred = pd.DataFrame(y_pred_np, index=y_test.index, columns=gene_cols)

    y_test = y_test.loc[rois,:]
    y_pred = y_pred.loc[rois,:]
    subset_metadata = subset_metadata.loc[rois,:]


    if same_scale:
        vmin1 = min(y_test[gene].min(), y_pred[gene].min())
        vmax1 = max(y_test[gene].max(), y_pred[gene].max())
        vmin2 = vmin1
        vmax2 = vmax1
    else:
        vmin1 = y_test[gene].min()
        vmax1 = y_test[gene].max()
        vmin2 = y_pred[gene].min()
        vmax2 = y_pred[gene].max()

    fig, axes = plt.subplots(1, 2, figsize=(14, 5)) 

    # Plot 1: Ground Truth
    sc1 = axes[0].imshow(he_img)
    scatter1 = axes[0].scatter(
        subset_metadata['scatter_x'], 
        subset_metadata['scatter_y'], 
        c=y_test[gene], cmap="seismic", alpha=1, vmin=vmin1,vmax=vmax1
    )
    axes[0].set_title(f"Ground Truth Expression of {gene}")
    axes[0].set_xlabel("X Coordinate")
    axes[0].set_ylabel("Y Coordinate")
    if ylim: axes[0].set_ylim(ylim)
    if xlim: axes[0].set_xlim(xlim)
    plt.colorbar(scatter1, ax=axes[0], label="Expression Level",shrink=0.75)

    # Plot 2: Prediction
    sc2 = axes[1].imshow(he_img)
    scatter2 = axes[1].scatter(
        subset_metadata['scatter_x'], 
        subset_metadata['scatter_y'], 
        c=y_pred[gene], cmap="seismic", alpha=1, vmin=vmin2,vmax=vmax2
    )
    axes[1].set_title(f"Predicted Expression of {gene}")
    axes[1].set_xlabel("X Coordinate")
    axes[1].set_ylabel("Y Coordinate")
    if ylim: axes[1].set_ylim(ylim)
    if xlim: axes[1].set_xlim(xlim)
    plt.colorbar(scatter2, ax=axes[1], label="Expression Level",shrink=0.75)

    plt.suptitle(f'Actual vs Predicted Gene Expression - {segment} {geneset} {gene}')
    plt.tight_layout()#
    
    plt.show()


In [ ]:
def plot_expr_compare(he_img, y_test, y_pred_np, rois, gene, subset_metadata, xlim=None, ylim=None):
   
    gene_cols = y_test.columns.drop("LEAP_ID") if "LEAP_ID" in y_test.columns else y_test.columns

    y_pred = pd.DataFrame(y_pred_np, index=y_test.index, columns=gene_cols)

    y_test = y_test.loc[rois,:]
    y_pred = y_pred.loc[rois,:]
    subset_metadata = subset_metadata.loc[rois,:]

    vmin1 = min(y_test[gene].min(), y_pred[gene].min())
    vmax1 = max(y_test[gene].max(), y_pred[gene].max())
    vmin2 = vmin1
    vmax2 = vmax1

    fig, axes = plt.subplots(1, 3, figsize=(14, 5)) 

    # Plot 1: Ground Truth
    sc1 = axes[0].imshow(he_img)
    scatter1 = axes[0].scatter(
        subset_metadata['scatter_x'], 
        subset_metadata['scatter_y'], 
        c=y_test[gene], cmap="seismic", alpha=1, vmin=vmin1,vmax=vmax1
    )
    axes[0].set_title(f"Ground Truth Expression of {gene}")
    axes[0].set_xlabel("X Coordinate")
    axes[0].set_ylabel("Y Coordinate")
    if ylim: axes[0].set_ylim(ylim)
    if xlim: axes[0].set_xlim(xlim)
    plt.colorbar(scatter1, ax=axes[0], label="Expression Level",shrink=0.75)

    # Plot 2: Prediction
    sc2 = axes[1].imshow(he_img)
    scatter2 = axes[1].scatter(
        subset_metadata['scatter_x'], 
        subset_metadata['scatter_y'], 
        c=y_pred[gene], cmap="seismic", alpha=1, vmin=vmin2,vmax=vmax2
    )
    axes[1].set_title(f"Predicted Expression of {gene}")
    axes[1].set_xlabel("X Coordinate")
    axes[1].set_ylabel("Y Coordinate")
    if ylim: axes[1].set_ylim(ylim)
    if xlim: axes[1].set_xlim(xlim)
    plt.colorbar(scatter2, ax=axes[1], label="Expression Level",shrink=0.75)

    # Plot 3: Prediction on own axis
    sc2 = axes[2].imshow(he_img)
    scatter2 = axes[2].scatter(
        subset_metadata['scatter_x'], 
        subset_metadata['scatter_y'], 
        c=y_pred[gene], cmap="seismic", alpha=1
    )
    axes[2].set_title(f"Predicted Expression of {gene}")
    axes[2].set_xlabel("X Coordinate")
    axes[2].set_ylabel("Y Coordinate")
    if ylim: axes[2].set_ylim(ylim)
    if xlim: axes[2].set_xlim(xlim)
    plt.colorbar(scatter2, ax=axes[2], label="Expression Level",shrink=0.75)

    plt.suptitle(f'Actual vs Predicted Gene Expression - {segment} {geneset} {gene}')
    plt.tight_layout()#
    
    plt.show()

In [ ]:

img_path = 'F:/01-data/LEAP/WSI/2-aligned/10x/cleaned'
OME_LEVEL = 2 # 10x mag on H&E
DOWNSAMPLE = 2**OME_LEVEL

In [ ]:

metadata_path = '../data/LEAP'
sample_metadata = pd.read_excel(os.path.join(metadata_path,"full_meta_data.xlsx"), index_col=0)

In [ ]:
sample_metadata["ROICoordinateX_downsampled"] = (sample_metadata["ROICoordinateX"] / DOWNSAMPLE).astype(int)
sample_metadata["ROICoordinateY_downsampled"] = (sample_metadata["ROICoordinateY"] / DOWNSAMPLE).astype(int)

### Genes of Interest

In [ ]:
genes_of_interest = {}
genes_of_interest[('CD45+','Panel')] = ['BCL2', 'IDO1', 'STAT1', 'CD274', 'CD3D', 'CXCL9'] #Panel
genes_of_interest[('CD45+','HVG')] = ['AREG','FDCSP','CPB1']
genes_of_interest[('CD45+','SVG')] = ['PRMT8','C7','TARP', 'TENT5C','RPL41','P2RY12','PITX1','CCSAP' ] #/SVG
genes_of_interest[('PanCK+','Panel')] = ['BRCA1', 'SNAI1', 'PIK3CA', 'BRCA2', 'EGFR'] #Panel
genes_of_interest[('PanCK+','HVG')] = ['IGHG1', 'IGHG4', 'IGHG3', 'IGKC',  'IGHG2',   'CSN3', 'CA9', 'CPB1' ]
genes_of_interest[('PanCK+','SVG')] = ['ATP5MC2', 'BRINP3', 'DMRTB1', 'NWD2', 'ENO1']

In [ ]:
geneset_good_models[geneset_good_models['Gene']=='EGFR']

In [ ]:
cd45_rois['LEAP_ID'].unique()

In [ ]:
panck_rois['LEAP_ID'].unique()

### Set LEAP ID

In [ ]:
leaporder =  ['LEAP120', 'LEAP121','LEAP065', 'LEAP079','LEAP082', 'LEAP093', 'LEAP064', 'LEAP045']

In [ ]:
limits = {}
limits['LEAP045'] = ((200,12000),(11600,5000))
limits['LEAP032'] = ((2200,5500),(2000,0))
limits['LEAP064'] = ((2300,11300),(19500,14500))
limits['LEAP065'] = ((2000,11000),(7500,2500))
limits['LEAP078'] = ((1200,5000),(7100,4500))
limits['LEAP079'] = ((0,11000),(7600,200))
limits['LEAP082'] = ((800,8200),(4100,0))
limits['LEAP093'] = ((3000,7500),(11000,8500))
limits['LEAP120'] = ((1700,11300),(22000,16700))
limits['LEAP121'] = ((1000,9100),(4500,0))

In [ ]:
LEAPID='LEAP120'

In [ ]:
leaprois = {}
leaprois['CD45+'] = list(cd45_rois[cd45_rois['LEAP_ID']==LEAPID].index)
leaprois['PanCK+'] = list(panck_rois[panck_rois['LEAP_ID']==LEAPID].index)

In [ ]:
downsample_factor=0.2
    # --- Adjust ROI coordinates to match downsampled image ---
sample_metadata.loc[:, 'scatter_x'] = sample_metadata.loc[:,"ROICoordinateY_downsampled"] * downsample_factor
sample_metadata.loc[:, 'scatter_y']  = sample_metadata.loc[:,"ROICoordinateX_downsampled"] * downsample_factor

In [ ]:
subset_metadata = sample_metadata[sample_metadata['LEAP_ID']==LEAPID]

In [ ]:
# Load H&E
heimgs = {}
for leap in leaporder:
    slide=get_slide_id(leap)
    he_path = os.path.join(img_path,f"{leap}_{slide.lower()}-aligned.tiff")
    he_img = Image.open(he_path)
    he_img = np.array(he_img)
    # --- Downsample image ---
    downsampled_img = resize(he_img, 
                            output_shape=(int(he_img.shape[0]*downsample_factor), 
                                        int(he_img.shape[1]*downsample_factor)),
                            anti_aliasing=True,
                            preserve_range=True).astype(he_img.dtype)
    heimgs[leap] = downsampled_img


In [ ]:
scaled_limits = {
    k: (
        tuple(int(x * downsample_factor) for x in v[0]),  # xlim
        tuple(int(y * downsample_factor) for y in v[1])   # ylim
    )
    for k, v in limits.items()
}
scaled_limits

Genes of Interest

In [ ]:
genelist_cd45 = ['BCL2','CPB1', 'IDO1', 'ST6GALNAC3', 'CD3D', 'FDCSP', 'C7', 'P2RY12']
genelist_panck = ['CPB1', 'ENO1']

In [ ]:

leaprois['PanCK+'] = list(panck_rois[panck_rois['LEAP_ID']==LEAPID].index)

## Figure 3 - Overlay

## Plot H&E

In [ ]:
from matplotlib.gridspec import GridSpec
from matplotlib.gridspec import GridSpecFromSubplotSpec
from matplotlib.patches import Rectangle

In [ ]:
def plot_overlay(leap_ids, gene, roi_df,y_test, y_pred, suffix="" ):

    fig = plt.figure(figsize=(10, 2.8 * len(leap_ids)), layout='constrained')
    gs = GridSpec(len(leap_ids), 4, width_ratios=[0.1, 1, 1, 0.05], figure=fig, wspace=0.05)

    n_rows = len(leap_ids)

    row_height = 1 / n_rows
    col = sns.color_palette("YlOrRd", as_cmap=True) 

    for i, leap_id in enumerate(leap_ids):
        rois = list(roi_df[roi_df['LEAP_ID'] == leap_id].index)
        values_gt = y_test.T.loc[rois, gene]
        values_pred = y_pred.T.loc[rois, gene]
        lim = scaled_limits[leap_id]
        image = heimgs[leap_id]

        gs_row = gs[i, :]  # This is a SubplotSpec
        sub_gs = GridSpecFromSubplotSpec(
                1, 4,
                subplot_spec=gs_row,
                width_ratios=[0.1, 1, 1, 0.05],
                wspace=0.05
            )
        plot_patient_row(
            fig,
            sub_gs,
            image, image,
            values_gt,
            values_pred,
            rois,
            sample_metadata,
            f"Patient {i+1}",
            lim, cmap=col,
            show_titles=(i == 0)  # Only show titles on first row
        )

        row_y = 1 - (i + 1) / n_rows
        # Adjust margins if needed (e.g., make left/right padding smaller)

    fig.suptitle(f'Expression of {gene}{suffix}', fontsize=SUPTITLESIZE)
    plt.show()

### Top Genes

y_test_cd45.T.loc[rois, gene]

plot_overlay(leap_ids, gene, roi_df=cd45_rois, 
            y_test=y_test_cd45, y_pred=y_pred_calib_cd45,
            suffix=" in Immune (CD45+) segments - CALIBRATED")

In [ ]:
leap_ids = ['LEAP120','LEAP121']
gene='PRMT8'

plot_overlay(leap_ids, gene, roi_df=cd45_rois, 
            y_test=y_test_cd45, y_pred=y_pred_calib_cd45,
            suffix=" in Immune (CD45+) segments - CALIBRATED")

In [ ]:
gene='BRCA1'
leap_ids = ['LEAP120','LEAP121']#,'LEAP064']#,'LEAP082','LEAP093']

plot_overlay(leap_ids, gene, roi_df=panck_rois, 
            y_test=y_test_panck, y_pred=y_pred_calib_panck,
            suffix=" in Epithelial (PanCK+) segments - CALIBRATED")

### Additional Genes

CD45

In [ ]:

top_genes_panck.sort_values('ConcordanceCorr', ascending=False).head()

In [ ]:

top_genes_cd45.sort_values('ConcordanceCorr', ascending=False).head()

In [ ]:

gene='C7'

plot_overlay(leap_ids, gene, roi_df=cd45_rois, 
            y_test=y_test_cd45, y_pred=y_pred_calib_cd45,
            suffix=" in Immune (CD45+) segments - CALIBRATED")

In [ ]:


#gene='ST6GALNAC3'
#gene='FDCSP'
#gene='PRMT8'
#gene='C7'
#gene='TENT5C'
gene='CD274'
leap_ids = ['LEAP120','LEAP121']#,'LEAP064']#,'LEAP082','LEAP093']

plot_overlay(leap_ids, gene, roi_df=cd45_rois, 
            y_test=y_test_cd45, y_pred=y_pred_cd45,
            suffix=" in Immune (CD45+) segments")


In [ ]:

leap_ids = ['LEAP120','LEAP121']#,'LEAP064']#,'LEAP082','LEAP093']

plot_overlay(leap_ids, gene, roi_df=cd45_rois, 
            y_test=y_test_cd45, y_pred=y_pred_calib_cd45,
            suffix=" in Immune (CD45+) segments - CALIBRATED")


In [ ]:
full_fe_order = ['Prov-Gigapath', 'UNI', 'Virchow','fused']

PanCK+

In [ ]:
gene='S100A2'

plot_overlay(leap_ids, gene, roi_df=panck_rois, 
            y_test=y_test_panck, y_pred=y_pred_calib_panck,
            suffix=" in Tumour (PanCK+) segments - CALIBRATED")

In [ ]:
gene='MKI67'
leap_ids = ['LEAP120','LEAP121']#,'LEAP064']#,'LEAP082','LEAP093']

plot_overlay(leap_ids, gene, roi_df=panck_rois, 
            y_test=y_test_panck, y_pred=y_pred_panck,
            suffix=" in Tumour (PanCK+) segments")

In [ ]:
gene='MKI67'
leap_ids = ['LEAP120','LEAP121']#,'LEAP064']#,'LEAP082','LEAP093']

plot_overlay(leap_ids, gene, roi_df=panck_rois, 
            y_test=y_test_panck, y_pred=y_pred_calib_panck,
            suffix=" in Tumour (PanCK+) segments - CALIBRATED")

ENO1 in first 2 rows
BRCA1

In [ ]:

gene='ENO1'
gene='CD274'
leap_ids = ['LEAP120','LEAP121']#,'LEAP064']#,'LEAP082','LEAP093']

plot_overlay(leap_ids, gene, roi_df=panck_rois, 
            y_test=y_test_panck, y_pred=y_pred_panck,
            suffix=" in Tumour (PanCK+) segments")

In [ ]:

leap_ids = ['LEAP120','LEAP121']#,'LEAP064']#,'LEAP082','LEAP093']

plot_overlay(leap_ids, gene, roi_df=panck_rois, 
            y_test=y_test_panck, y_pred=y_pred_calib_panck,
            suffix=" in Tumour (PanCK+) segments - CALIBRATED")

## Plot ROIs with registration

In [ ]:
from visualize import plot_rois_on_image

In [ ]:
img_path = 'F:/01-data/LEAP/WSI'

In [ ]:
# Load H&E
LEAPID='LEAP121'
SLIDE_ID=get_slide_id(LEAPID)
he_path = os.path.join(img_path,'2-aligned','10x','cleaned',f"{LEAPID}_{SLIDE_ID.lower()}-aligned.tiff")
print(he_path)
he_img_reg = Image.open(he_path)
he_img_reg = np.array(he_img_reg)
he_img_reg.shape

Slide_48 _LEAP121

In [ ]:
# Load H&E original
he_path = os.path.join(img_path,'1-transformations','10x-tif',f"{SLIDE_ID.upper()}_{LEAPID}.tif")
he_path = os.path.join(img_path,'1-transformations','10x-tif',f"{SLIDE_ID}_{LEAPID}.tif")
he_img_orig = Image.open(he_path)
he_img_orig = np.array(he_img_orig)
he_img_orig.shape

In [ ]:
SLIDE_ID

In [ ]:
# Load H&E original
path='F:/01-data/LEAP/OME-TIFF/1-converted/10x'
if_path = os.path.join(path,f"{LEAPID}-{SLIDE_ID}.tif")
if_img = Image.open(if_path)
if_img = np.array(if_img)
if_img.shape

In [ ]:
subset_metadata = sample_metadata[sample_metadata['LEAP_ID']==LEAPID]

{'ROI:0': {'x': 1874, 'y': 539, 'width': 282, 'height': 282, 'name': 'Roi'},
roi num: x:ROICoordinateX_downsampled, y:ROICoordinateY_downsampled, w, h, name

In [ ]:
subset_metadata.head(3)

In [ ]:
def plot_rois(img, metadata, ax=None, limits=None, title='ROIs Overlaid on Image',lw=2):

    rect_w=244
    ax.imshow(img)



    #for roi_id, rect in roi_dict.items():
    for x, y, roi_id in zip(metadata["ROICoordinateY_downsampled"],
                            metadata["ROICoordinateX_downsampled"],
                            metadata["Roi"]):
        # Extract rectangle data
        
        #print(f"roi {roi_id} : {x}, {y}")
        # Create a rectangle patch
        rectangle = patches.Rectangle(
            (x - rect_w/2, y - rect_w/2),  # Bottom-left corner
            rect_w,   # Width
            rect_w,  # Height
            linewidth=lw,
            edgecolor='blue',
            facecolor='none'
        )
        # Add the rectangle to the plot
        ax.add_patch(rectangle)

        # Add the ROI ID as a label
        ax.text(
            x,  # Center of the rectangle (x)
            y,  # Center of the rectangle (y)
            #roi_id,          # ROI ID label
            int(roi_id),          # ROI ID label
            color='red',
            fontsize=7,
            ha='center',     # Horizontal alignment
            va='center'      # Vertical alignment
        )

    # Customize the plot
    ax.set_title(title, fontsize=24)
    ax.set_xlabel('X Coordinate')
    ax.set_ylabel('Y Coordinate')
    if limits: ax.set_xlim(limits[0])
    if limits: ax.set_ylim(limits[1])
    #ax.set_aspect('equal')  # Keep aspect ratio equal
    ax.set_aspect('auto')
    ax.axis('off')
    #plt.gca().invert_yaxis()  # Invert y-axis to align with image coordinate system

    return ax
